# Random Forest - Cross Validation (FHS)

Notebook para executar Random Forest com StratifiedKFold sobre o dataset FHS.
- Selecione qual dataset (raw ou normalizado) usar alterando a variável `DATASET_PATH` no bloco de configuração.
- Os parâmetros do modelo estão em `rf_params`.
- Outputs: `model_reports/random_forest/cv/` contém `csv/`, `images`, `pdf`.
- O modelo será salvo em `databases/FHS/common/models/random_forest/v1/rf_model_cf{N_SPLITS}.pkl`. 

In [1]:
# Configuração
import os
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

BASE = Path('../../')  # ajustável se necessário
# Use o dataset raw de FHS por padrão; EDA pode gerar versões normalizadas se desejar
# Nome do dataset cru: fhs_framingham.csv
DATASET_NAME = 'fhs_framingham.csv'  # ou stroke_standard_scaled.csv / stroke_robust_scaled.csv / stroke_minmax_scaled.csv
# Preferir carregar do raw; se quiser usar versões processadas, ajuste a linha abaixo
DATASET_PATH = Path('../data/processed')/ DATASET_NAME
# Se quiser forçar explicitamente qual coluna é o target (no bruto ou no normalizado), defina aqui
# Para FHS normalmente a coluna alvo é 'stroke' (0/1)
TARGET_COLUMN = 'TenYearCHD' # ajustar se necessário

# Output paths
REPORTS = Path('../model_reports/random_forest/cv')
CSV_OUT = REPORTS / 'csv'
IMAGES_OUT = REPORTS / 'images'
PDF_OUT = REPORTS / 'pdf'
MODEL_DIR = Path('../common/models/random_forest/v1')
# Basico outputs (single train/test) - sibling folder to 'cv'
BASICO_DIR = REPORTS.parent / 'basico'
BASICO_CSV = BASICO_DIR / 'csv'
BASICO_IMAGES = BASICO_DIR / 'images'

for d in [REPORTS, CSV_OUT, IMAGES_OUT, PDF_OUT, MODEL_DIR, BASICO_DIR, BASICO_CSV, BASICO_IMAGES]:
    d.mkdir(parents=True, exist_ok=True)

# Random Forest params (variável conforme solicitado)
rf_params = {
    "n_estimators": 650,
    "max_depth": 16,
    "min_samples_split": 8,
    "min_samples_leaf": 15,
    "max_features": 0.3,
    "bootstrap": True,
    "class_weight": "balanced_subsample",
    "criterion": "entropy",
    'n_jobs': 8
}

# CV and threshold config
N_SPLITS = 10
THRESHOLD = 0.5
# Colunas identificadoras que NÃO devem entrar como features no modelo, mas devem ser mantidas nos arquivos de saída
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', 'stroke', 'TenYearCHD']

# Pareto configuration: threshold (fraction) and selected-features placeholder (set to None to be updated after CV)
PARETO_THRESHOLD = 0.9  # fraction, e.g. 0.8 == 80%
PARETO_SELECTED_FEATURES = None

# Variance percent-difference threshold (fraction). Example: 0.1 == 10% difference.
VAR_DIFF_PCT_THRESHOLD = 0.01
TARGET_COLUMN = 'TenYearCHD'  # ajustar se necessário
TEST_SIZE = 0.3
RT_RANDOM_STATE = 42
VARIAVEIS_CATEGORICAS = ['male', 'education', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'prevalentHyp', 'diabetes']


In [ ]:
# Imports principais
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (accuracy_score, f1_score, roc_auc_score, precision_score, recall_score,
                             balanced_accuracy_score, confusion_matrix, roc_curve, log_loss, auc)
from matplotlib.backends.backend_pdf import PdfPages
from tqdm.auto import tqdm
import joblib
from sklearn.metrics import log_loss
import matplotlib.pyplot as plt
sns.set(style='whitegrid')

In [3]:
# Load dataset (path definido na configuração)
import shutil
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')

# Preserve a raw copy of the original CSV (unaltered) in data/processed with suffix _raw
RAW_COPY_PATH = Path('../data/processed') / f"{DATASET_NAME.replace('.csv','')}_raw.csv"
try:
    shutil.copy2(DATASET_PATH, RAW_COPY_PATH)
    print(f'Raw dataset copiado para: {RAW_COPY_PATH}')
except Exception as e_copy:
    print('Não foi possível copiar o CSV original para processed:', e_copy)

# Leia o dataset (continuamos a usar df para processamento posterior)
df = pd.read_csv(DATASET_PATH)
# df['y'] = df['stroke']
# Função utilitária para encontrar coluna aproximada (case/strip tolerant)
def find_column(df, name):
    # tenta match exato
    if name in df.columns:
        return name
    # tenta match case-insensitive
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    # tenta encontrar coluna que contenha o nome
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None

# Determina a coluna alvo: usa TARGET_COLUMN quando fornecida, senão infere
target_col = None
if TARGET_COLUMN:
    candidate = find_column(df, TARGET_COLUMN)
    if candidate is None:
        raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada. Colunas: {list(df.columns)}")
    target_col = candidate
else:
    # heurística leve: nomes comuns
    for cand in ['y', 'stroke', 'target', 'label', 'class']:
        candidate = find_column(df, cand)
        if candidate:
            target_col = candidate
            break
    # procura colunas binárias
    if target_col is None:
        for c in df.columns:
            if c.lower().strip() in ('id', 'patient_id', 'pid'):
                continue
            try:
                nunq = df[c].nunique(dropna=True)
            except Exception:
                nunq = 999
            if nunq == 2:
                target_col = c
                break
    # fallback para última coluna com poucos valores distintos
    if target_col is None:
        lastc = df.columns[-1]
        if df[lastc].nunique(dropna=True) <= 10:
            target_col = lastc
    if target_col is None:
        raise ValueError(f"Não foi possível inferir a coluna alvo. Defina TARGET_COLUMN ou verifique as colunas: {list(df.columns)}")

# Renomeia para 'y' e garante coluna disponível antes de acessar
if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if 'y' not in df.columns:
    raise KeyError('Coluna y não encontrada após renomeação')

# Se y não for numérico, tenta converter diretamente (para FHS, deve ser 0/1)
if not np.issubdtype(df['y'].dtype, np.number):
    # factorize para inteiros 0..k-1
    df['y'], uniques = pd.factorize(df['y'])
    # salva versão discretizada para inspeção
    df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_y_numeric.csv', index=False)

# Substitui inf valores e dropna (pode ajustar imputação se preferir)
df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

# reorganiza para garantir que y seja a última coluna
cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

# salva uma cópia de entrada no csv de reports (final)
df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)

print('Dataset shape:', df.shape)

Raw dataset copiado para: ..\data\processed\fhs_framingham_raw.csv
Dataset shape: (3180, 17)


In [4]:
df

,id,male,age,education,currentSmoker,cigsPerDay,BPMeds,prevalentStroke,prevalentHyp,diabetes,totChol,sysBP,diaBP,BMI,heartRate,glucose,y
0,1,1,39,4.0,0,0.0,0.0,0,0,0,195.0,106.0,70.0,26.97,80.0,77.0,0
1,2,0,46,2.0,0,0.0,0.0,0,0,0,250.0,121.0,81.0,28.73,95.0,76.0,0
2,3,1,48,1.0,1,20.0,0.0,0,0,0,245.0,127.5,80.0,25.34,75.0,70.0,0
3,4,0,61,3.0,1,30.0,0.0,0,1,0,225.0,150.0,95.0,28.58,65.0,103.0,1
4,5,0,46,3.0,1,23.0,0.0,0,0,0,285.0,130.0,84.0,23.10,85.0,85.0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3175,4234,1,50,1.0,1,1.0,0.0,0,1,0,313.0,179.0,92.0,25.97,66.0,86.0,1
3176,4235,1,51,3.0,1,43.0,0.0,0,0,0,207.0,126.5,80.0,19.71,65.0,68.0,0
3177,4238,0,52,2.0,0,0.0,0.0,0,0,0,269.0,133.5,83.0,21.47,80.0,107.0,0
3178,4239,1,40,3.0,0,0.0,0.0,0,1,0,185.0,141.0,98.0,25.60,67.0,72.0,0


In [ ]:
# ================== OPTUNA RF (8/1/1) + FEATURE SELECTION + RELATÓRIO (VALIDACAO/TESTE) =====OTIMIZA AUC THRESHOLD FIXO=============
import warnings, joblib
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
from optuna.exceptions import TrialPruned
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# ---------- CONFIG ----------
N_TRIALS   = 30
SEED       = 42
FS_THRESH  = 1.0
MIN_FEATURES_KEPT = 15

SAVE_DIR   = Path("../model_reports/random_forest_optuna")
SAVE_DIR.mkdir(parents=True, exist_ok=True)

EXCLUDE_COLUMNS = EXCLUDE_COLUMNS if "EXCLUDE_COLUMNS" in globals() else []

# ======================================================================
#                            UTILIDADES
# ======================================================================
def split_feature_sets(df, exclude_cols, target="y"):
    exclude_lower = {c.lower() for c in exclude_cols}
    id_cols = [c for c in df.columns if c.lower() in exclude_lower and c != target]
    features = [c for c in df.columns if c not in id_cols + [target]]
    num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in features if c not in num_cols]
    return id_cols, features, num_cols, cat_cols

def preprocess_fold(X_tr_df, X_te_df, cat_cols, num_cols, features_order):
    # categóricas
    if cat_cols:
        X_tr_cat = X_tr_df[cat_cols].astype("object").fillna("Unknown")
        X_te_cat = X_te_df[cat_cols].astype("object").fillna("Unknown")
        oenc = OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-1
        )
        X_tr_cat_enc = pd.DataFrame(
            oenc.fit_transform(X_tr_cat),
            columns=cat_cols,
            index=X_tr_df.index
        )
        X_te_cat_enc = pd.DataFrame(
            oenc.transform(X_te_cat),
            columns=cat_cols,
            index=X_te_df.index
        )
    else:
        X_tr_cat_enc = pd.DataFrame(index=X_tr_df.index)
        X_te_cat_enc = pd.DataFrame(index=X_te_df.index)

    # numéricas
    if num_cols:
        X_tr_num = X_tr_df[num_cols].apply(pd.to_numeric, errors="coerce")
        X_te_num = X_te_df[num_cols].apply(pd.to_numeric, errors="coerce")
        med = X_tr_num.median(numeric_only=True)
        X_tr_num = X_tr_num.fillna(med)
        X_te_num = X_te_num.fillna(med)
    else:
        X_tr_num = pd.DataFrame(index=X_tr_df.index)
        X_te_num = pd.DataFrame(index=X_te_df.index)

    X_tr_proc = pd.concat([X_tr_num, X_tr_cat_enc], axis=1)[features_order]
    X_te_proc = pd.concat([X_te_num, X_te_cat_enc], axis=1)[features_order]
    return X_tr_proc, X_te_proc

def stratified_8_1_1_indices(y, seed=SEED):
    # 80% train e 20% temp
    sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=seed)
    tr_idx_full, temp_idx = next(sss1.split(np.zeros(len(y)), y))
    y_temp = y.iloc[temp_idx]
    # 10% val e 10% test dentro dos 20%
    sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=seed)
    val_rel_idx, te_rel_idx = next(sss2.split(np.zeros(len(y_temp)), y_temp))
    val_idx = temp_idx[val_rel_idx]
    te_idx  = temp_idx[te_rel_idx]
    return tr_idx_full, val_idx, te_idx

def compute_feature_importance(df_features_only, features, y_train, seed=SEED):
    """
    Feature importance APENAS com dados do TREINO.
    """
    sss = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=seed)
    importances = np.zeros(len(features), dtype=float)

    # detecta tipos no DF original
    num_cols_all = [c for c in features if pd.api.types.is_numeric_dtype(df_features_only[c])]
    cat_cols_all = [c for c in features if c not in num_cols_all]

    for tr_idx, te_idx in sss.split(df_features_only[features], y_train):
        X_tr = df_features_only.iloc[tr_idx][features].copy()
        X_te = df_features_only.iloc[te_idx][features].copy()
        y_tr = y_train.iloc[tr_idx].copy()

        X_tr_proc, X_te_proc = preprocess_fold(X_tr, X_te, cat_cols_all, num_cols_all, features)

        rf = RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            n_jobs=-1,
            random_state=seed,
            bootstrap=True
        )
        rf.fit(X_tr_proc, y_tr)
        importances += rf.feature_importances_

    importances /= 5.0
    fi = pd.Series(importances, index=features).sort_values(ascending=False)
    return fi

def suggest_params(trial: optuna.Trial, seed=SEED):
    n_estimators      = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth         = trial.suggest_categorical("max_depth", [None] + list(range(4, 41)))
    min_samples_split = trial.suggest_int("min_samples_split", 2, 30)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 30)
    bootstrap         = trial.suggest_categorical("bootstrap", [True, False])
    class_weight      = trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"])

    mf_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "fraction"])
    if mf_mode == "fraction":
        max_features = trial.suggest_float("max_features", 0.2, 0.9, step=0.05)
    else:
        max_features = mf_mode

    params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        class_weight=class_weight,
        max_features=max_features,
        n_jobs=-1,
        random_state=seed
    )
    if bootstrap:
        params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)

    # guardamos max_features_mode no best_params e depois normalizamos
    params["max_features_mode"] = mf_mode
    return params

def normalize_best_params(best_params: dict, seed=SEED):
    bp = best_params.copy()
    mf_mode = bp.pop("max_features_mode", None)

    if mf_mode == "fraction":
        # já existe bp["max_features"] numérico
        pass
    else:
        # max_features estava como 'sqrt'/'log2' em bp["max_features"]
        pass

    bp["n_jobs"] = -1
    bp["random_state"] = seed
    return bp

def auc_general(y_true, proba, n_classes):
    if n_classes == 2:
        return float(roc_auc_score(y_true, proba[:, 1]))
    return float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro"))

def cross_entropy_per_class_multiclass(y_true, proba, eps=1e-12, max_classes=3):
    """
    Retorna CE por classe (0..max_classes-1).
    Se a classe não existir no y_true, retorna NaN para ela.
    Se o problema tiver menos classes do que max_classes, classes extras = NaN.
    """
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    proba = np.clip(proba, eps, 1 - eps)

    n_classes = proba.shape[1]
    ce = [np.nan] * max_classes

    for c in range(min(n_classes, max_classes)):
        mask = (y_true == c)
        if np.any(mask):
            ce[c] = float(np.mean(-np.log(proba[mask, c])))
    return ce  # [CE-C0, CE-C1, CE-C2]

def confusion_percent_binary(y_true, y_pred):
    """
    Para binário: retorna TP%, FP%, TN%, FN% (percentual do total).
    """
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN

    if total == 0:
        return np.nan, np.nan, np.nan, np.nan

    return TP/total, FP/total, TN/total, FN/total

def all_metrics_layout(y_true, proba, threshold=0.5, eps=1e-12):
    """
    Retorna exatamente as colunas do seu layout:
    ACC, F1, BAC, PRE, REC, AUC, CE-C0, CE-C1, CE-C2, TP%, FP%, TN%, FN%
    """
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    n_classes = proba.shape[1]

    # pred
    if n_classes == 2:
        y_pred = (proba[:, 1] >= threshold).astype(int)
    else:
        y_pred = np.argmax(proba, axis=1)

    # métricas clássicas
    acc = float(accuracy_score(y_true, y_pred))
    bac = float(balanced_accuracy_score(y_true, y_pred))

    if n_classes == 2:
        f1  = float(f1_score(y_true, y_pred, zero_division=0))
        pre = float(precision_score(y_true, y_pred, zero_division=0))
        rec = float(recall_score(y_true, y_pred, zero_division=0))
        auc = float(roc_auc_score(y_true, proba[:, 1]))
        tp_pct, fp_pct, tn_pct, fn_pct = confusion_percent_binary(y_true, y_pred)
    else:
        # macro para multiclasse
        f1  = float(f1_score(y_true, y_pred, average="macro", zero_division=0))
        pre = float(precision_score(y_true, y_pred, average="macro", zero_division=0))
        rec = float(recall_score(y_true, y_pred, average="macro", zero_division=0))
        auc = float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro"))

        # TP/FP/TN/FN não é bem definido em multiclass sem “um-vs-rest”
        # Para manter o layout, deixamos NaN.
        tp_pct, fp_pct, tn_pct, fn_pct = np.nan, np.nan, np.nan, np.nan

    # cross entropy por classe (até C2)
    ce0, ce1, ce2 = cross_entropy_per_class_multiclass(y_true, proba, eps=eps, max_classes=3)

    return {
        "ACC": acc,
        "F1": f1,
        "BAC": bac,
        "PRE": pre,
        "REC": rec,
        "AUC": auc,
        "CE-C0": ce0,
        "CE-C1": ce1,
        "CE-C2": ce2,
        "TP%": tp_pct,
        "FP%": fp_pct,
        "TN%": tn_pct,
        "FN%": fn_pct,
    }

# ======================================================================
#                            DATA
# ======================================================================
if "y" not in df.columns:
    raise KeyError("A coluna-alvo 'y' precisa existir em df.")

y = df["y"].copy()
id_cols, FEATURES_ALL, _, _ = split_feature_sets(df, EXCLUDE_COLUMNS, target="y")

tr_idx, val_idx, te_idx = stratified_8_1_1_indices(y, seed=SEED)

X_train = df.iloc[tr_idx][FEATURES_ALL].copy()
y_train = y.iloc[tr_idx].copy()
X_val   = df.iloc[val_idx][FEATURES_ALL].copy()
y_val   = y.iloc[val_idx].copy()
X_test  = df.iloc[te_idx][FEATURES_ALL].copy()
y_test  = y.iloc[te_idx].copy()

# ======================================================================
#                       FEATURE SELECTION (TREINO)
# ======================================================================
fi = compute_feature_importance(X_train, FEATURES_ALL, y_train, seed=SEED)
fi_norm = fi / (fi.sum() if fi.sum() != 0 else 1.0)
cum = fi_norm.cumsum()
selected = cum[cum <= FS_THRESH].index.tolist()
if len(selected) < MIN_FEATURES_KEPT:
    selected = fi.index[:MIN_FEATURES_KEPT].tolist()

print(f"🔎 Total de features: {len(FEATURES_ALL)} | Selecionadas ({FS_THRESH*100:.0f}%): {len(selected)}")

# ======================================================================
#                          OPTUNA (AUC) - CV INTERNA
# ======================================================================
def objective_builder(features, setup_name):
    num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in features if c not in num_cols]
    inner_kf = StratifiedKFold(n_splits=8, shuffle=True, random_state=SEED)

    n_classes = len(np.unique(y_train))

    def objective(trial: optuna.Trial) -> float:
        params = suggest_params(trial, seed=SEED)
        # tira max_features_mode do dict passado ao RF
        rf_params = params.copy()
        rf_params.pop("max_features_mode", None)

        scores = []
        for fold_id, (tr_i, va_i) in enumerate(inner_kf.split(X_train, y_train), start=1):
            X_tr = X_train.iloc[tr_i][features].copy()
            y_tr = y_train.iloc[tr_i].copy()
            X_va = X_train.iloc[va_i][features].copy()
            y_va = y_train.iloc[va_i].copy()

            X_tr_proc, X_va_proc = preprocess_fold(X_tr, X_va, cat_cols, num_cols, features)

            model = RandomForestClassifier(**rf_params)
            model.fit(X_tr_proc, y_tr)

            proba = model.predict_proba(X_va_proc)
            score = auc_general(y_va, proba, n_classes=n_classes)

            scores.append(score)
            trial.report(float(np.mean(scores)), step=fold_id)
            if trial.should_prune():
                raise TrialPruned()

        return float(np.mean(scores))

    return objective

print("\n🚀 Optuna (AUC CV interna) – SELECTED FEATURES (por padrão)")
study = optuna.create_study(
    direction="maximize",
    sampler=TPESampler(seed=SEED),
    pruner=MedianPruner(n_startup_trials=20, n_warmup_steps=2),
    study_name="rf_auc_selected_features"
)
study.optimize(objective_builder(selected, "selected_features"), n_trials=N_TRIALS, show_progress_bar=True)

best_params = normalize_best_params(study.best_params, seed=SEED)
# remove chave auxiliar se sobrou
best_params.pop("max_features_mode", None)

print("\n✅ Melhor AUC (CV interna):", f"{study.best_value:.6f}")
print("✅ Melhores hiperparâmetros:", best_params)

# ======================================================================
#                   TREINA E AVALIA (VALIDACAO e TESTE)
# ======================================================================
def fit_predict_proba(features, params, X_fit, y_fit, X_eval):
    num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df[c])]
    cat_cols = [c for c in features if c not in num_cols]

    X_fit_proc, X_eval_proc = preprocess_fold(
        X_fit[features].copy(),
        X_eval[features].copy(),
        cat_cols, num_cols, features
    )
    model = RandomForestClassifier(**params)
    model.fit(X_fit_proc, y_fit)
    return model.predict_proba(X_eval_proc)

# threshold só faz sentido para binário
THRESHOLD = 0.5
EPS = 1e-12

# (A) validação: treina no treino (80) e avalia no val (10)
proba_val = fit_predict_proba(selected, best_params, X_train, y_train, X_val)
val_metrics = all_metrics_layout(y_val, proba_val, threshold=THRESHOLD, eps=EPS)

# (B) teste: treina no treino+val (90) e avalia no teste (10)
X_trv = pd.concat([X_train, X_val], axis=0)
y_trv = pd.concat([y_train, y_val], axis=0)
proba_test = fit_predict_proba(selected, best_params, X_trv, y_trv, X_test)
test_metrics = all_metrics_layout(y_test, proba_test, threshold=THRESHOLD, eps=EPS)

# ======================================================================
#                 ARQUIVO FINAL NO LAYOUT QUE VOCÊ PEDIU
# ======================================================================
layout_cols = ["ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","TP%","FP%","TN%","FN%"]

df_layout = pd.DataFrame(
    [
        {"split": "VALIDACAO", **val_metrics},
        {"split": "TESTES", **test_metrics},
    ]
).set_index("split")[layout_cols]

# salva
out_csv = SAVE_DIR / "cv_811_validation_test_layout.csv"
df_layout.to_csv(out_csv, index=True)

# imprime bonito
pd.options.display.float_format = lambda x: f"{x:.6f}"
print("\n===== LAYOUT FINAL (VALIDACAO / TESTES) =====")
print(df_layout.to_string())

print("\n💾 Arquivo salvo em:", out_csv.resolve())


[I 2026-01-26 04:12:10,719] A new study created in memory with name: rf_auc_selected_features


🔎 Total de features: 15 | Selecionadas (100%): 15

🚀 Optuna (AUC CV interna) – SELECTED FEATURES (por padrão)


Best trial: 0. Best value: 0.688948:   3%|▎         | 1/30 [00:16<08:09, 16.87s/it]

[I 2026-01-26 04:12:27,584] Trial 0 finished with value: 0.6889477825707604 and parameters: {'n_estimators': 500, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}. Best is trial 0 with value: 0.6889477825707604.


Best trial: 1. Best value: 0.700943:   7%|▋         | 2/30 [00:40<09:39, 20.69s/it]

...

[I 2026-01-26 04:17:58,683] Trial 27 finished with value: 0.7058051098154738 and parameters: {'n_estimators': 200, 'max_depth': 34, 'min_samples_split': 7, 'min_samples_leaf': 30, 'bootstrap': True, 'class_weight': 'balanced', 'max_features_mode': 'log2', 'max_samples': 0.5938218171250059}. Best is trial 27 with value: 0.7058051098154738.


Best trial: 27. Best value: 0.705805:  97%|█████████▋| 29/30 [05:53<00:06,  6.20s/it]

[I 2026-01-26 04:18:03,902] Trial 28 finished with value: 0.7033940061455228 and parameters: {'n_estimators': 200, 'max_depth': 34, 'min_samples_split': 7, 'min_samples_leaf': 30, 'bootstrap': True, 'class_weight': 'balanced', 'max_features_mode': 'log2', 'max_samples': 0.6166255525663275}. Best is trial 27 with value: 0.7058051098154738.


Best trial: 27. Best value: 0.705805: 100%|██████████| 30/30 [06:00<00:00, 12.03s/it]


[I 2026-01-26 04:18:11,527] Trial 29 finished with value: 0.6930772543797064 and parameters: {'n_estimators': 250, 'max_depth': 7, 'min_samples_split': 24, 'min_samples_leaf': 6, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.9, 'max_samples': 0.5855748467811447}. Best is trial 27 with value: 0.7058051098154738.

✅ Melhor AUC (CV interna): 0.705805
✅ Melhores hiperparâmetros: {'n_estimators': 200, 'max_depth': 34, 'min_samples_split': 7, 'min_samples_leaf': 30, 'bootstrap': True, 'class_weight': 'balanced', 'max_samples': 0.5938218171250059, 'n_jobs': -1, 'random_state': 42}

===== LAYOUT FINAL (VALIDACAO / TESTES) =====
               ACC       F1      BAC      PRE      REC      AUC    CE-C0    CE-C1  CE-C2      TP%      FP%      TN%      FN%
split                                                                                                                       
VALIDACAO 0.698113 0.323944 0.629260 0.232323 0.534884 0.685835 0.537569 0.7

In [ ]:
#Novo teste porque estava usando hold out simples e agora estou fazendo com k-fold cross validation.

In [10]:
# ================== SCRIPT COMPLETO COM SUAS CONFIGURAÇÕES DE DIRETÓRIO =====================
import warnings, joblib
import numpy as np
import pandas as pd
from pathlib import Path

import optuna
from optuna.exceptions import TrialPruned
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.model_selection import StratifiedShuffleSplit, StratifiedKFold
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    roc_auc_score,
    balanced_accuracy_score,
    accuracy_score,
    f1_score,
    precision_score,
    recall_score
)
from sklearn.ensemble import RandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)

# ======================================================================
#                            SUAS CONFIGURAÇÕES E LEITURA DE DADOS
# ======================================================================
BASE = Path('../../')
DATASET_NAME = 'fhs_framingham.csv'
DATASET_PATH = BASE / Path('fhs/data/processed') / DATASET_NAME
TARGET_COLUMN = 'TenYearCHD'  # Coluna alvo

REPORTS = BASE / Path('model_reports/random_forest/cv_optuna_kfold')
CSV_OUT = REPORTS / 'csv'
MODEL_DIR = BASE / Path('common/models/random_forest/v2_kfold')

for d in [REPORTS, CSV_OUT, MODEL_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# Configurações do tuning Optuna/RF
N_TRIALS = 30
SEED = 42
MIN_FEATURES_KEPT = 15
N_FOLDS_TUNING = 10
TEST_SIZE_FINAL = 0.1
EXCLUDE_COLUMNS = ['id', 'ID', 'patient_id', 'pid', 'stroke', 'TenYearCHD']  # ignorar como features


def find_column(df, name):
    if name in df.columns:
        return name
    low = {c.lower().strip(): c for c in df.columns}
    if name.lower().strip() in low:
        return low[name.lower().strip()]
    for c in df.columns:
        if name.lower().strip() in c.lower():
            return c
    return None


# Carregamento e limpeza do DataFrame
if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Arquivo de dataset não encontrado: {DATASET_PATH.resolve()}')
df = pd.read_csv(DATASET_PATH)

target_col = find_column(df, TARGET_COLUMN)
if target_col is None:
    raise ValueError(f"TARGET_COLUMN='{TARGET_COLUMN}' não encontrada.")

if target_col != 'y':
    df = df.rename(columns={target_col: 'y'})
if not np.issubdtype(df['y'].dtype, np.number):
    df['y'], uniques = pd.factorize(df['y'])

df.replace([np.inf, -np.inf], np.nan, inplace=True)
df = df.dropna()

cols = [c for c in df.columns if c != 'y'] + ['y']
df = df[cols]

df.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}', index=False)
print('Dataset shape após limpeza:', df.shape)


# ======================================================================
#                            UTILIDADES
# ======================================================================
def split_feature_sets(df_in, exclude_cols, target="y"):
    exclude_lower = {c.lower() for c in exclude_cols}
    id_cols = [c for c in df_in.columns if c.lower() in exclude_lower and c != target]
    features = [c for c in df_in.columns if c not in id_cols + [target]]
    num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df_in[c])]
    cat_cols = [c for c in features if c not in num_cols]
    return id_cols, features, num_cols, cat_cols


def preprocess_fold(X_tr_df, X_te_df, cat_cols, num_cols, features_order):
    # Categóricas
    if cat_cols:
        X_tr_cat = X_tr_df[cat_cols].astype("object").fillna("Unknown")
        X_te_cat = X_te_df[cat_cols].astype("object").fillna("Unknown")
        oenc = OrdinalEncoder(handle_unknown="use_encoded_value",
                              unknown_value=-1, encoded_missing_value=-1)
        X_tr_cat_enc = pd.DataFrame(oenc.fit_transform(X_tr_cat), columns=cat_cols, index=X_tr_df.index)
        X_te_cat_enc = pd.DataFrame(oenc.transform(X_te_cat), columns=cat_cols, index=X_te_df.index)
    else:
        X_tr_cat_enc = pd.DataFrame(index=X_tr_df.index)
        X_te_cat_enc = pd.DataFrame(index=X_te_df.index)

    # Numéricas
    if num_cols:
        X_tr_num = X_tr_df[num_cols].apply(pd.to_numeric, errors="coerce")
        X_te_num = X_te_df[num_cols].apply(pd.to_numeric, errors="coerce")
        med = X_tr_num.median(numeric_only=True)
        X_tr_num, X_te_num = X_tr_num.fillna(med), X_te_num.fillna(med)
    else:
        X_tr_num = pd.DataFrame(index=X_tr_df.index)
        X_te_num = pd.DataFrame(index=X_te_df.index)

    X_tr_proc = pd.concat([X_tr_num, X_tr_cat_enc], axis=1)[features_order]
    X_te_proc = pd.concat([X_te_num, X_te_cat_enc], axis=1)[features_order]
    return X_tr_proc, X_te_proc


def stratified_tv_te_indices(y, seed=SEED, test_size=0.1):
    sss = StratifiedShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    tv_idx, te_idx = next(sss.split(np.zeros(len(y)), y))
    return tv_idx, te_idx


def compute_feature_importance(df_features_only, features, y_train, seed=SEED):
    sss = StratifiedShuffleSplit(n_splits=5, test_size=0.2, random_state=seed)
    importances = np.zeros(len(features), dtype=float)

    num_cols_all = [c for c in features if pd.api.types.is_numeric_dtype(df_features_only[c])]
    cat_cols_all = [c for c in features if c not in num_cols_all]

    for tr_idx, te_idx in sss.split(df_features_only[features], y_train):
        X_tr = df_features_only.iloc[tr_idx][features].copy()
        X_te = df_features_only.iloc[te_idx][features].copy()
        y_tr = y_train.iloc[tr_idx].copy()

        X_tr_proc, X_te_proc = preprocess_fold(X_tr, X_te, cat_cols_all, num_cols_all, features)

        rf = RandomForestClassifier(
            n_estimators=300, max_depth=None,
            n_jobs=-1, random_state=seed, bootstrap=True
        )
        rf.fit(X_tr_proc, y_tr)
        importances += rf.feature_importances_

    importances /= 5.0
    return pd.Series(importances, index=features).sort_values(ascending=False)


def suggest_params(trial: optuna.Trial, seed=SEED):
    n_estimators = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth = trial.suggest_categorical("max_depth", [None] + list(range(4, 41)))
    min_samples_split = trial.suggest_int("min_samples_split", 2, 30)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 30)
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])
    class_weight = trial.suggest_categorical("class_weight", [None, "balanced", "balanced_subsample"])
    mf_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "fraction"])
    max_features = trial.suggest_float("max_features", 0.2, 0.9, step=0.05) if mf_mode == "fraction" else mf_mode

    params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        bootstrap=bootstrap,
        class_weight=class_weight,
        max_features=max_features,
        n_jobs=-1,
        random_state=seed,
        max_features_mode=mf_mode
    )
    if bootstrap:
        params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)
    return params


def normalize_best_params(best_params: dict, seed=SEED):
    bp = best_params.copy()
    bp.pop("max_features_mode", None)
    if bp.get("bootstrap") is False and "max_samples" in bp:
        del bp["max_samples"]
    bp["n_jobs"], bp["random_state"] = -1, seed
    return bp


def auc_general(y_true, proba, n_classes):
    if n_classes == 2:
        return float(roc_auc_score(y_true, proba[:, 1]))
    return float(roc_auc_score(y_true, proba, multi_class="ovr", average="macro"))


def cross_entropy_per_class_multiclass(y_true, proba, eps=1e-12, max_classes=3):
    y_true = np.asarray(y_true).astype(int)
    proba = np.asarray(proba)
    proba = np.clip(proba, eps, 1 - eps)
    n_classes = proba.shape[1]
    ce = [np.nan] * max_classes
    for c in range(min(n_classes, max_classes)):
        mask = (y_true == c)
        if np.any(mask):
            ce[c] = float(np.mean(-np.log(proba[mask, c])))
    return ce


def confusion_percent_binary(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)
    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN
    if total == 0:
        return np.nan, np.nan, np.nan, np.nan
    return TP / total, FP / total, TN / total, FN / total


def all_metrics_layout(y_true, proba, threshold=0.5, eps=1e-12):
    y_true = np.asarray(y_true).astype(int)
    y_pred = (proba[:, 1] >= threshold).astype(int)

    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, zero_division=0)
    bac = balanced_accuracy_score(y_true, y_pred)
    pre = precision_score(y_true, y_pred, zero_division=0)
    rec = recall_score(y_true, y_pred, zero_division=0)

    auc = auc_general(y_true, proba, 2)

    # ---- CORREÇÃO MÍNIMA AQUI: não desempacotar 3 valores quando max_classes=2
    ce_list = cross_entropy_per_class_multiclass(y_true, proba, eps=eps, max_classes=2)
    ce_c0 = ce_list[0]
    ce_c1 = ce_list[1]

    tp_p, fp_p, tn_p, fn_p = confusion_percent_binary(y_true, y_pred)

    return pd.Series({
        'ACC': acc, 'F1': f1, 'BAC': bac, 'PRE': pre, 'REC': rec, 'AUC': auc,
        'CE-C0': ce_c0, 'CE-C1': ce_c1, 'CE-C2': np.nan,
        'TP%': tp_p, 'FP%': fp_p, 'TN%': tn_p, 'FN%': fn_p
    })


# ======================================================================
#                            OBJETIVO OPTUNA (K-FOLD INTERNO)
# ======================================================================
def objective(trial: optuna.Trial, X_train_full, y_train_full, features, cat_cols, num_cols, target_classes):
    params = suggest_params(trial)
    params_norm = normalize_best_params(params)
    if params_norm.get("bootstrap") is False and "max_samples" in params_norm:
        del params_norm["max_samples"]

    skf = StratifiedKFold(n_splits=N_FOLDS_TUNING, shuffle=True, random_state=SEED)
    auc_scores = []

    for tr_idx, val_idx in skf.split(X_train_full, y_train_full):
        X_tr = X_train_full.iloc[tr_idx].copy()
        y_tr = y_train_full.iloc[tr_idx].copy()
        X_val = X_train_full.iloc[val_idx].copy()
        y_val = y_train_full.iloc[val_idx].copy()

        X_tr_proc, X_val_proc = preprocess_fold(X_tr, X_val, cat_cols, num_cols, features)

        model = RandomForestClassifier(**params_norm)
        model.fit(X_tr_proc, y_tr)

        y_val_proba = model.predict_proba(X_val_proc)

        try:
            auc = auc_general(y_val, y_val_proba, target_classes)
            auc_scores.append(auc)
        except ValueError:
            raise TrialPruned()

    return float(np.mean(auc_scores))


# ======================================================================
#                            EXECUÇÃO PRINCIPAL
# ======================================================================
if __name__ == "__main__":

    # 1) Split TV(90%) / Teste(10%)
    tv_idx, te_idx = stratified_tv_te_indices(df["y"], SEED, test_size=TEST_SIZE_FINAL)
    df_tv = df.iloc[tv_idx].copy()
    df_te = df.iloc[te_idx].copy()

    # 2) Features (somente TV)
    id_cols, features, num_cols, cat_cols = split_feature_sets(df_tv, EXCLUDE_COLUMNS)
    X_train_full = df_tv[features]
    y_train_full = df_tv["y"]
    target_classes = len(np.unique(y_train_full))

    X_test_final = df_te[features]
    y_test_final = df_te["y"]

    # 3) Feature Importance (TV)
    print("Calculando Feature Importance...")
    feature_importance_series = compute_feature_importance(X_train_full, features, y_train_full, SEED)
    features_kept = feature_importance_series.index[:MIN_FEATURES_KEPT].tolist()
    features = features_kept
    print(f"Features selecionadas: {features}")

    # ---- AJUSTE IMPORTANTE: recalcule num_cols/cat_cols após reduzir features
    num_cols = [c for c in features if pd.api.types.is_numeric_dtype(df_tv[c])]
    cat_cols = [c for c in features if c not in num_cols]

    # Atualiza X com as features selecionadas
    X_train_full = df_tv[features]
    X_test_final = df_te[features]

    # 4) Optuna tuning com CV interno
    print("Iniciando Tuning com Optuna e 10-Fold CV interno...")
    sampler = TPESampler(seed=SEED)
    pruner = MedianPruner(n_startup_trials=5, n_warmup_steps=0, interval_steps=1)
    study = optuna.create_study(direction="maximize", sampler=sampler, pruner=pruner)

    study.optimize(
        lambda trial: objective(trial, X_train_full, y_train_full, features, cat_cols, num_cols, target_classes),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )

    # 5) Treino final (TV) + Teste isolado
    print(f"\nMelhor AUC de Validação (Média K-Fold): {study.best_value:.4f}")
    best_params = normalize_best_params(study.best_params, SEED)

    print("Treinando modelo final nos dados de Treino/Validação (90%)...")
    X_tr_proc_final, X_te_proc_final = preprocess_fold(
        X_train_full, X_test_final, cat_cols, num_cols, features
    )
    final_model = RandomForestClassifier(**best_params)
    final_model.fit(X_tr_proc_final, y_train_full)

    print("Avaliando no conjunto de Teste ISOLADO (10%)...")
    y_test_proba = final_model.predict_proba(X_te_proc_final)
    test_metrics = all_metrics_layout(y_test_final, y_test_proba)

    report_df = pd.DataFrame(test_metrics, columns=["TESTE"]).T
    print("\n===== LAYOUT FINAL (TESTE ISOLADO COM K-FOLD TUNING) =====")
    print(report_df.to_markdown())

    joblib.dump(final_model, MODEL_DIR / "final_model.pkl")
    report_df.to_csv(CSV_OUT / "final_report.csv")
    print(f"\nModelo e relatório salvos em:\nModelo: {MODEL_DIR}\nRelatório: {CSV_OUT}")


Dataset shape após limpeza: (3180, 17)
Calculando Feature Importance...


[I 2026-02-08 15:21:44,345] A new study created in memory with name: no-name-88f63ec6-a844-40e4-80f1-c16f66b8df27


Features selecionadas: ['sysBP', 'age', 'BMI', 'totChol', 'diaBP', 'glucose', 'heartRate', 'cigsPerDay', 'education', 'male', 'prevalentHyp', 'currentSmoker', 'BPMeds', 'prevalentStroke', 'diabetes']
Iniciando Tuning com Optuna e 10-Fold CV interno...


Best trial: 0. Best value: 0.686513:   3%|▎         | 1/30 [00:12<06:04, 12.58s/it]

[I 2026-02-08 15:21:56,920] Trial 0 finished with value: 0.6865128858125462 and parameters: {'n_estimators': 500, 'max_depth': 13, 'min_samples_split': 14, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.30000000000000004, 'max_samples': 0.9847923138822793}. Best is trial 0 with value: 0.6865128858125462.


...

[I 2026-02-08 15:32:38,850] Trial 27 finished with value: 0.698304927952636 and parameters: {'n_estimators': 750, 'max_depth': 11, 'min_samples_split': 9, 'min_samples_leaf': 30, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'log2', 'max_samples': 0.5119009523920975}. Best is trial 21 with value: 0.6993339427974402.


Best trial: 21. Best value: 0.699334:  97%|█████████▋| 29/30 [11:11<00:20, 20.65s/it]

[I 2026-02-08 15:32:55,544] Trial 28 finished with value: 0.697073483958034 and parameters: {'n_estimators': 550, 'max_depth': 18, 'min_samples_split': 2, 'min_samples_leaf': 20, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'fraction', 'max_features': 0.9, 'max_samples': 0.5481879523636396}. Best is trial 21 with value: 0.6993339427974402.


Best trial: 21. Best value: 0.699334: 100%|██████████| 30/30 [11:35<00:00, 23.18s/it]


[I 2026-02-08 15:33:19,813] Trial 29 finished with value: 0.6932461037830308 and parameters: {'n_estimators': 950, 'max_depth': 39, 'min_samples_split': 12, 'min_samples_leaf': 7, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'sqrt', 'max_samples': 0.5030101220384147}. Best is trial 21 with value: 0.6993339427974402.

Melhor AUC de Validação (Média K-Fold): 0.6993
Treinando modelo final nos dados de Treino/Validação (90%)...
Avaliando no conjunto de Teste ISOLADO (10%)...

===== LAYOUT FINAL (TESTE ISOLADO COM K-FOLD TUNING) =====
|       |      ACC |   F1 |   BAC |   PRE |   REC |      AUC |    CE-C0 |   CE-C1 |   CE-C2 |   TP% |   FP% |      TN% |      FN% |
|:------|---------:|-----:|------:|------:|------:|---------:|---------:|--------:|--------:|------:|------:|---------:|---------:|
| TESTE | 0.867925 |    0 |   0.5 |     0 |     0 | 0.767512 | 0.135717 | 1.72467 |     nan |     0 |     0 | 0.867925 | 0.132075 |

Modelo e relatório salvos em:
Modelo: ..\..\common

In [ ]:
#Novo teste porque estava usando hold out simples e agora estou fazendo com k-fold cross validation.

In [12]:
# ============================ 8/1/1 NESTED-STYLE COM OPTUNA (RF) =========OTIMIZA ACURACIA TRHESHOLD AJUSTAVEL DESEMPENHOU MELHOR===================
# OBJETIVO: otimizar ACURÁCIA (na CV interna) e escolher AUTOMATICAMENTE o MELHOR THRESHOLD
# - Sem vazamento: imputer dentro do Pipeline; threshold escolhido APENAS na validação (fold 9)
# - Saídas:
#   cv_811_inner_folds.csv
#   cv_811_validation.csv
#   cv_811_test.csv
#   cv_811_results_long.csv
#   cv_811_report_layout.csv   (layout: VALIDACAO/TESTES x colunas ACC..FN%)

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    roc_auc_score, accuracy_score, f1_score, precision_score, recall_score,
    balanced_accuracy_score, average_precision_score
)
from sklearn.ensemble import RandomForestClassifier

# -------------------- CONFIG --------------------
RANDOM_STATE     = 42
N_TRIALS         = 10
N_SPLITS_OUTER   = 10
N_SPLITS_INNER   = 8
EPS              = 1e-12

# grid de thresholds para busca automática
THR_GRID = np.round(np.linspace(0.01, 0.99, 99), 2)

# -------------------- ENTRADAS ESPERADAS --------------------
# df: DataFrame com coluna alvo 'y'
# EXCLUDE_COLUMNS: lista de colunas de ID etc (opcional)
EXCLUDE_COLUMNS = EXCLUDE_COLUMNS if "EXCLUDE_COLUMNS" in globals() else []
if "df" not in globals():
    raise RuntimeError("⚠️ Você precisa ter um DataFrame 'df' no ambiente.")
if "y" not in df.columns:
    raise KeyError("⚠️ A coluna alvo 'y' precisa existir em df.")

# ---------- separa X,y sem vazamento ----------
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
id_cols = [c for c in df.columns if c.lower() in exclude_lower and c.lower() != "y"]
FEATURES = [c for c in df.columns if c not in id_cols + ["y"]]

X_raw = df[FEATURES].copy()
y     = df["y"].copy()

# Se o alvo vier como {0,1} ou strings, padroniza para int quando possível
# (para multi-classe, mantemos como está)
if y.nunique() == 2:
    # tenta mapear para 0/1
    uniq = list(pd.unique(y))
    if set(uniq) != {0, 1}:
        # ordena e mapeia
        uniq_sorted = sorted(uniq)
        y = y.map({uniq_sorted[0]: 0, uniq_sorted[1]: 1}).astype(int)

# ---------- define outer folds (10) e fixa 8/1/1 ----------
outer_kf = StratifiedKFold(n_splits=N_SPLITS_OUTER, shuffle=True, random_state=RANDOM_STATE)
folds = [test_idx for _, test_idx in outer_kf.split(X_raw, y)]

train8_idx = np.concatenate(folds[:8])
val1_idx   = folds[8]
test1_idx  = folds[9]

X_train8, y_train8 = X_raw.iloc[train8_idx], y.iloc[train8_idx]
X_val1,   y_val1   = X_raw.iloc[val1_idx],   y.iloc[val1_idx]
X_test1,  y_test1  = X_raw.iloc[test1_idx],  y.iloc[test1_idx]

# ===================== Threshold automático (melhor por acurácia) =====================
def choose_best_threshold_by_accuracy(y_true, p_pos, grid=THR_GRID):
    """
    Escolhe o threshold que maximiza ACC.
    Desempate: maior Recall; depois threshold mais próximo de 0.5
    """
    y_true = np.asarray(y_true).astype(int)
    p_pos  = np.asarray(p_pos)

    best = {"thr": 0.5, "acc": -1.0, "rec": -1.0, "dist05": 1e9}
    for thr in grid:
        y_pred = (p_pos >= thr).astype(int)
        acc = accuracy_score(y_true, y_pred)
        rec = recall_score(y_true, y_pred, zero_division=0)
        dist05 = abs(thr - 0.5)
        if (acc > best["acc"]) or (acc == best["acc"] and rec > best["rec"]) or (acc == best["acc"] and rec == best["rec"] and dist05 < best["dist05"]):
            best = {"thr": float(thr), "acc": float(acc), "rec": float(rec), "dist05": float(dist05)}
    return best["thr"]

# ===================== Métricas =====================
def cross_entropy_per_class_general(y_true, proba, eps=1e-12):
    """
    Retorna CE-C0, CE-C1, CE-C2 (quando existir; senão NaN).
    - Binário: proba pode ser p_pos (n,) ou matriz (n,2)
    - Multiclasse: proba deve ser (n,K)
    """
    y_true = np.asarray(y_true)
    proba = np.asarray(proba)

    # binário: se vier vetor p_pos, converte para (n,2)
    if proba.ndim == 1:
        p1 = np.clip(proba, eps, 1 - eps)
        p0 = 1 - p1
        proba = np.column_stack([p0, p1])

    proba = np.clip(proba, eps, 1 - eps)

    classes = np.unique(y_true)
    # garante índices 0..K-1 se y_true já é 0..K-1
    # caso contrário, cria mapeamento
    if not np.array_equal(classes, np.arange(len(classes))):
        mapping = {c:i for i,c in enumerate(classes)}
        y_idx = np.array([mapping[v] for v in y_true], dtype=int)
    else:
        y_idx = y_true.astype(int)

    ces = []
    K = proba.shape[1]
    for k in range(min(3, K)):
        mask = (y_idx == k)
        if np.any(mask):
            ce_k = float(np.mean(-np.log(proba[mask, k])))
        else:
            ce_k = np.nan
        ces.append(ce_k)

    # completa até 3
    while len(ces) < 3:
        ces.append(np.nan)

    return ces[0], ces[1], ces[2]

def binary_confusion_percentages(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int)
    y_pred = np.asarray(y_pred).astype(int)

    TP = int(np.sum((y_true == 1) & (y_pred == 1)))
    TN = int(np.sum((y_true == 0) & (y_pred == 0)))
    FP = int(np.sum((y_true == 0) & (y_pred == 1)))
    FN = int(np.sum((y_true == 1) & (y_pred == 0)))
    total = TP + TN + FP + FN
    if total == 0:
        return TP, FP, TN, FN, np.nan, np.nan, np.nan, np.nan
    return TP, FP, TN, FN, TP/total, FP/total, TN/total, FN/total

def metrics_row(name, y_true, proba_or_p_pos, threshold=None):
    """
    Retorna um dict com:
    ACC, F1, BAC, PRE, REC, AUC, CE-C0, CE-C1, CE-C2, TP%, FP%, TN%, FN%, Threshold
    - Para binário: threshold obrigatório para métricas de classe; AUC usa probas
    - Para multiclasse: threshold e TP%/FP%/TN%/FN% ficam NaN
    """
    y_true_arr = np.asarray(y_true)
    proba = np.asarray(proba_or_p_pos)

    out = {"Dataset": name}

    if y_true.nunique() == 2:
        # garante p_pos
        if proba.ndim == 2 and proba.shape[1] >= 2:
            p_pos = proba[:, 1]
        else:
            p_pos = proba

        thr = 0.5 if threshold is None else float(threshold)
        y_pred = (p_pos >= thr).astype(int)

        out["Threshold"] = thr
        out["ACC"] = float(accuracy_score(y_true_arr, y_pred))
        out["F1"]  = float(f1_score(y_true_arr, y_pred, zero_division=0))
        out["BAC"] = float(balanced_accuracy_score(y_true_arr, y_pred))
        out["PRE"] = float(precision_score(y_true_arr, y_pred, zero_division=0))
        out["REC"] = float(recall_score(y_true_arr, y_pred, zero_division=0))

        # AUC e AUPRC (AUPRC não entra no layout pedido, mas ajuda no debug)
        out["AUC"]  = float(roc_auc_score(y_true_arr, p_pos))
        out["AUPRC"] = float(average_precision_score(y_true_arr, p_pos))

        ce0, ce1, ce2 = cross_entropy_per_class_general(y_true_arr, p_pos, eps=EPS)
        out["CE-C0"], out["CE-C1"], out["CE-C2"] = ce0, ce1, ce2

        TP, FP, TN, FN, TPp, FPp, TNp, FNp = binary_confusion_percentages(y_true_arr, y_pred)
        out["TP"] = TP; out["FP"] = FP; out["TN"] = TN; out["FN"] = FN
        out["TP%"] = float(TPp); out["FP%"] = float(FPp); out["TN%"] = float(TNp); out["FN%"] = float(FNp)
    else:
        # multiclasse
        out["Threshold"] = np.nan
        y_pred = np.argmax(proba, axis=1)
        out["ACC"] = float(accuracy_score(y_true_arr, y_pred))
        out["F1"]  = float(f1_score(y_true_arr, y_pred, average="macro", zero_division=0))
        out["BAC"] = float(balanced_accuracy_score(y_true_arr, y_pred))
        out["PRE"] = float(precision_score(y_true_arr, y_pred, average="macro", zero_division=0))
        out["REC"] = float(recall_score(y_true_arr, y_pred, average="macro", zero_division=0))
        try:
            out["AUC"] = float(roc_auc_score(y_true_arr, proba, multi_class="ovr", average="macro"))
        except Exception:
            out["AUC"] = np.nan
        ce0, ce1, ce2 = cross_entropy_per_class_general(y_true_arr, proba, eps=EPS)
        out["CE-C0"], out["CE-C1"], out["CE-C2"] = ce0, ce1, ce2
        out["TP%"] = np.nan; out["FP%"] = np.nan; out["TN%"] = np.nan; out["FN%"] = np.nan

    return out

# ===================== OPTUNA (otimizando ACURÁCIA) =====================
import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

inner_kf = StratifiedKFold(n_splits=N_SPLITS_INNER, shuffle=True, random_state=RANDOM_STATE)

def build_model(trial):
    n_estimators      = trial.suggest_int("n_estimators", 200, 1000, step=50)
    max_depth_choice  = trial.suggest_categorical("max_depth_choice", ["None", "int"])
    max_depth         = None if max_depth_choice == "None" else trial.suggest_int("max_depth", 3, 50, step=1)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20, step=1)
    min_samples_leaf  = trial.suggest_int("min_samples_leaf", 1, 20, step=1)
    bootstrap         = trial.suggest_categorical("bootstrap", [True, False])
    class_weight      = trial.suggest_categorical("class_weight", [None, "balanced"])
    max_features_mode = trial.suggest_categorical("max_features_mode", ["sqrt", "log2", "float"])
    if max_features_mode == "float":
        max_features = trial.suggest_float("max_features_float", 0.2, 1.0)
    else:
        max_features = max_features_mode

    rf_params = dict(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        class_weight=class_weight,
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
    if bootstrap:
        rf_params["max_samples"] = trial.suggest_float("max_samples", 0.5, 1.0)

    clf = Pipeline(steps=[
        ("imp", SimpleImputer(strategy="median")),
        ("model", RandomForestClassifier(**rf_params))
    ])
    return clf

def objective(trial):
    clf = build_model(trial)
    accs = []

    for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
        X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
        y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

        clf.fit(X_tr, y_tr)

        # otimização de threshold "dentro" do fold (sem olhar val1/test1)
        proba_va = clf.predict_proba(X_va)[:, 1]
        thr_fold = choose_best_threshold_by_accuracy(y_va, proba_va, grid=THR_GRID)

        y_pred_va = (proba_va >= thr_fold).astype(int)
        acc = accuracy_score(y_va, y_pred_va)
        accs.append(float(acc))

        trial.report(float(acc), step=fold_id)
        if trial.should_prune():
            raise optuna.TrialPruned()

    return float(np.mean(accs))

study = optuna.create_study(
    study_name="rf_optuna_8fold_search_optimize_accuracy",
    direction="maximize",
    sampler=TPESampler(seed=RANDOM_STATE, n_startup_trials=10),
    pruner=MedianPruner(n_startup_trials=8, n_warmup_steps=2),
)

study.optimize(objective, n_trials=N_TRIALS, timeout=None, show_progress_bar=True)

# ---------- reconstrói best_params ----------
best = study.best_trial
bp = dict(best.params)
if bp.get("max_depth_choice") == "None":
    bp["max_depth"] = None
bp["max_features"] = bp.get("max_features_float", bp.get("max_features_mode"))
for k in ["max_depth_choice", "max_features_mode", "max_features_float"]:
    bp.pop(k, None)
if not bp.get("bootstrap", True):
    bp.pop("max_samples", None)
bp["n_jobs"] = -1
bp["random_state"] = RANDOM_STATE
rf_best_params = bp

print("\nMelhor ACC (CV interna 8 folds):", best.value)
print("Melhores hiperparâmetros:", rf_best_params)

# ===================== (1) Reavalia CV interna (8 folds) com THR otimizado por fold =====================
inner_fold_rows = []
inner_clf = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])

for fold_id, (tr_idx, va_idx) in enumerate(inner_kf.split(X_train8, y_train8), start=1):
    X_tr, X_va = X_train8.iloc[tr_idx], X_train8.iloc[va_idx]
    y_tr, y_va = y_train8.iloc[tr_idx], y_train8.iloc[va_idx]

    inner_clf.fit(X_tr, y_tr)
    proba_va = inner_clf.predict_proba(X_va)[:, 1]
    thr_fold = choose_best_threshold_by_accuracy(y_va, proba_va, grid=THR_GRID)

    row = metrics_row("inner_cv", y_va, proba_va, threshold=thr_fold)
    row.update({"etapa": "inner_cv", "fold": fold_id, "modo": "thr_best_acc_fold"})
    inner_fold_rows.append(row)

df_inner = pd.DataFrame(inner_fold_rows)
df_inner.to_csv("cv_811_inner_folds.csv", index=False)

# ===================== (2) Validação (fold 9) + threshold escolhido aqui =====================
clf_val = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])

clf_val.fit(X_train8, y_train8)
proba_val = clf_val.predict_proba(X_val1)[:, 1]

thr_star = choose_best_threshold_by_accuracy(y_val1, proba_val, grid=THR_GRID)
print(f"\n✅ Threshold escolhido no fold 9 (best ACC): {thr_star:.4f}")

val_row = metrics_row("VALIDACAO", y_val1, proba_val, threshold=thr_star)
val_row.update({"etapa": "validacao", "fold": 9, "modo": "thr_best_acc_val"})
df_val = pd.DataFrame([val_row])
df_val.to_csv("cv_811_validation.csv", index=False)

# ===================== (3) Treino final (8+1) e teste (fold 10) usando O MESMO threshold =====================
X_train9 = pd.concat([X_train8, X_val1], axis=0)
y_train9 = pd.concat([y_train8, y_val1], axis=0)

clf_final = Pipeline(steps=[
    ("imp", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(**rf_best_params))
])

clf_final.fit(X_train9, y_train9)
proba_test = clf_final.predict_proba(X_test1)[:, 1]

test_row = metrics_row("TESTES", y_test1, proba_test, threshold=thr_star)
test_row.update({"etapa": "teste", "fold": 10, "modo": "thr_from_val"})
df_test = pd.DataFrame([test_row])
df_test.to_csv("cv_811_test.csv", index=False)

# ===================== (4) Consolida tudo (long) =====================
df_all = pd.concat([df_inner, df_val, df_test], ignore_index=True)
df_all.to_csv("cv_811_results_long.csv", index=False)

# ===================== (5) Arquivo com o LAYOUT que você pediu =====================
# layout:
#           ACC F1 BAC PRE REC AUC CE-C0 CE-C1 CE-C2 TP% FP% TN% FN%
layout_cols = ["ACC","F1","BAC","PRE","REC","AUC","CE-C0","CE-C1","CE-C2","TP%","FP%","TN%","FN%"]

def _pick_layout_row(df_one_row):
    return {
        "ACC":  df_one_row["ACC"],
        "F1":   df_one_row["F1"],
        "BAC":  df_one_row["BAC"],
        "PRE":  df_one_row["PRE"],
        "REC":  df_one_row["REC"],
        "AUC":  df_one_row["AUC"],
        "CE-C0": df_one_row["CE-C0"],
        "CE-C1": df_one_row["CE-C1"],
        "CE-C2": df_one_row["CE-C2"],
        "TP%":  df_one_row["TP%"],
        "FP%":  df_one_row["FP%"],
        "TN%":  df_one_row["TN%"],
        "FN%":  df_one_row["FN%"],
    }

layout_df = pd.DataFrame(
    [
        {"SPLIT": "VALIDACAO", **_pick_layout_row(df_val.iloc[0].to_dict())},
        {"SPLIT": "TESTES",    **_pick_layout_row(df_test.iloc[0].to_dict())},
    ]
).set_index("SPLIT")[layout_cols]

layout_df.to_csv("cv_811_report_layout.csv")

# ===================== Impressão na tela (o “plugin” que você pediu) =====================
pd.options.display.float_format = lambda x: f"{x:.6f}"

print("\n===== RELATÓRIO (layout) =====")
print(layout_df.to_string())

print("\n===== Threshold final usado (val/test) =====")
print(f"threshold = {thr_star:.6f}")

print("\n===== Linhas completas (val/test) =====")
print(df_all[df_all["etapa"].isin(["validacao","teste"])].to_string(index=False))

print("\nArquivos gerados:")
print(" - cv_811_inner_folds.csv")
print(" - cv_811_validation.csv")
print(" - cv_811_test.csv")
print(" - cv_811_results_long.csv")
print(" - cv_811_report_layout.csv")


[I 2026-01-26 04:21:29,236] A new study created in memory with name: rf_optuna_8fold_search_optimize_accuracy
Best trial: 0. Best value: 0.870283:  10%|█         | 1/10 [00:14<02:09, 14.34s/it]

[I 2026-01-26 04:21:43,576] Trial 0 finished with value: 0.8702830188679245 and parameters: {'n_estimators': 500, 'max_depth_choice': 'None', 'min_samples_split': 13, 'min_samples_leaf': 4, 'bootstrap': True, 'class_weight': None, 'max_features_mode': 'float', 'max_features_float': 0.8659541126403374, 'max_samples': 0.6061695553391381}. Best is trial 0 with value: 0.8702830188679245.


Best trial: 1. Best value: 0.871069:  20%|██        | 2/10 [00:28<01:53, 14.14s/it]

[I 2026-01-26 04:21:57,566] Trial 1 finished with value: 0.8710691823899371 and parameters: {'n_estimators': 350, 'max_depth_choice': 'int', 'max_depth': 28, 'min_samples_split': 10, 'min_samples_leaf': 6, 'bootstrap': True, 'class_weight': 'balanced', 'max_features_mode': 'log2', 'max_samples': 0.7571172192068059}. Best is trial 1 with value: 0.8710691823899371.


Best trial: 1. Best value: 0.871069:  30%|███       | 3/10 [00:49<02:01, 17.33s/it]

...

[I 2026-01-26 04:23:49,485] Trial 7 finished with value: 0.8698899371069182 and parameters: {'n_estimators': 600, 'max_depth_choice': 'None', 'min_samples_split': 2, 'min_samples_leaf': 3, 'bootstrap': False, 'class_weight': 'balanced', 'max_features_mode': 'sqrt'}. Best is trial 1 with value: 0.8710691823899371.


Best trial: 1. Best value: 0.871069:  90%|█████████ | 9/10 [02:28<00:15, 15.15s/it]

[I 2026-01-26 04:23:58,158] Trial 8 pruned. 


Best trial: 1. Best value: 0.871069: 100%|██████████| 10/10 [02:49<00:00, 16.99s/it]


[I 2026-01-26 04:24:19,155] Trial 9 finished with value: 0.8694968553459119 and parameters: {'n_estimators': 950, 'max_depth_choice': 'None', 'min_samples_split': 6, 'min_samples_leaf': 9, 'bootstrap': False, 'class_weight': 'balanced', 'max_features_mode': 'sqrt'}. Best is trial 1 with value: 0.8710691823899371.

Melhor ACC (CV interna 8 folds): 0.8710691823899371
Melhores hiperparâmetros: {'n_estimators': 350, 'max_depth': 28, 'min_samples_split': 10, 'min_samples_leaf': 6, 'bootstrap': True, 'class_weight': 'balanced', 'max_samples': 0.7571172192068059, 'max_features': 'log2', 'n_jobs': -1, 'random_state': 42}

✅ Threshold escolhido no fold 9 (best ACC): 0.7000

===== RELATÓRIO (layout) =====
               ACC       F1      BAC      PRE      REC      AUC    CE-C0    CE-C1  CE-C2      TP%      FP%      TN%      FN%
SPLIT                                                                                                                       
VALIDACAO 0.871069 0.088889 0.523256 1.000000

In [ ]:
rf_params

In [ ]:
# ================== IMPORTS ==================
import os, time, joblib
import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import OrdinalEncoder
from sklearn.metrics import (
    accuracy_score, f1_score, balanced_accuracy_score, precision_score, recall_score,
    roc_auc_score, roc_curve, confusion_matrix, log_loss
)
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from matplotlib.backends.backend_pdf import PdfPages
from tqdm import tqdm

# ================== CONFIGURAÇÃO ==================
# Fração de features a manter (ex.: 0.6 = 60%). Use None ou 1.0 para desativar.
FEATURE_SUBSET_FRAC = 1

# Agora usamos split único 8/1/1
N_SPLITS = 1
RANDOM_STATE = 42

# Caso THRESHOLD não esteja definido no ambiente global, define padrão binário.
if 'THRESHOLD' not in globals():
    THRESHOLD = 0.5

# ================== PREPARAÇÃO X, y ==================
y = df['y']

# Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
ID_COLS = [c for c in ID_COLS if c != 'y']  # garante que não inclua a target

# FEATURES são todas as colunas exceto ids e y
FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]
X_full = df[ID_COLS + FEATURES] if ID_COLS else df[FEATURES]

# Detecta tipos (numéricas x categóricas) a partir de FEATURES
num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
cat_cols = [c for c in FEATURES if c not in num_cols]

# ---------- Split estratificado 8/1/1 ----------
sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_STATE)
train_idx_full, temp_idx = next(sss1.split(df[FEATURES], y))
y_temp = y.iloc[temp_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.5, random_state=RANDOM_STATE)
val_rel_idx, test_rel_idx = next(sss2.split(df.iloc[temp_idx][FEATURES], y_temp))
val_idx = temp_idx[val_rel_idx]
test_idx = temp_idx[test_rel_idx]

# ================== HELPERS ==================
def select_top_features(X_tr: pd.DataFrame, y_tr: pd.Series, frac: float, random_state: int = 42):
    """
    Seleciona top-k features por importância usando ExtraTrees (rápido e robusto).
    Treina SOMENTE no fold de treino (sem vazamento).
    """
    if frac is None or not (0 < frac < 1.0):
        return list(X_tr.columns)  # sem redução
    n_total = X_tr.shape[1]
    k = max(1, int(np.ceil(frac * n_total)))
    ranker = ExtraTreesClassifier(
        n_estimators=200, max_features='sqrt', n_jobs=-1, random_state=random_state
    )
    ranker.fit(X_tr, y_tr)
    importances = pd.Series(ranker.feature_importances_, index=X_tr.columns)
    keep = importances.sort_values(ascending=False).head(k).index.tolist()
    return keep

# ================== ESTRUTURAS DE RESULTADOS ==================
resultados = []
aurocs_train, aurocs_test = [], []
fprs_train, tprs_train = [], []
fprs_test,  tprs_test  = [], []
importancias_lista = []

X_train_parts, X_test_parts = [], []
y_train_parts, y_test_parts = [], []
y_train_pred_parts, y_test_pred_parts = [], []

start_total = time.time()

# ================== LOOP ÚNICO (8/1/1) ==================
with PdfPages(PDF_OUT / f'RF_CV_{N_SPLITS}.pdf') as pdf:
    with tqdm(total=N_SPLITS, desc="🔄 Split 8/1/1") as pbar:
        # simulamos a estrutura de "fold" (apenas 1)
        for fold, (train_idx, _test_idx) in enumerate([(train_idx_full, test_idx)], 1):
            t0 = time.time()

            # Partições completas (mantendo IDs p/ rastreabilidade)
            X_train_full = X_full.iloc[train_idx].copy()
            X_val_full   = X_full.iloc[val_idx].copy()
            X_test_full  = X_full.iloc[_test_idx].copy()

            # Apenas as FEATURES para modelagem
            X_train = X_train_full[FEATURES].copy()
            X_val   = X_val_full[FEATURES].copy()
            X_test  = X_test_full[FEATURES].copy()

            y_train = y.iloc[train_idx]
            y_val   = y.iloc[val_idx]
            y_test_ = y.iloc[_test_idx]

            # --------- PRÉ-PROCESSAMENTO (SEM VAZAMENTO) ---------
            # 1) Categóricas → OrdinalEncoder (ajustado no treino)
            if cat_cols:
                X_train_cat = X_train[cat_cols].astype('object').fillna('Unknown')
                X_val_cat   = X_val[cat_cols].astype('object').fillna('Unknown')
                X_test_cat  = X_test[cat_cols].astype('object').fillna('Unknown')

                oenc = OrdinalEncoder(
                    handle_unknown='use_encoded_value',
                    unknown_value=-1,
                    encoded_missing_value=-1
                )
                X_train_cat_enc = pd.DataFrame(
                    oenc.fit_transform(X_train_cat),
                    columns=cat_cols, index=X_train.index
                )
                X_val_cat_enc = pd.DataFrame(
                    oenc.transform(X_val_cat),
                    columns=cat_cols, index=X_val.index
                )
                X_test_cat_enc = pd.DataFrame(
                    oenc.transform(X_test_cat),
                    columns=cat_cols, index=X_test.index
                )
            else:
                X_train_cat_enc = pd.DataFrame(index=X_train.index)
                X_val_cat_enc   = pd.DataFrame(index=X_val.index)
                X_test_cat_enc  = pd.DataFrame(index=X_test.index)

            # 2) Numéricas → coerção + imputação por mediana do TREINO
            if num_cols:
                X_train_num = X_train[num_cols].apply(pd.to_numeric, errors='coerce')
                X_val_num   = X_val[num_cols].apply(pd.to_numeric, errors='coerce')
                X_test_num  = X_test[num_cols].apply(pd.to_numeric, errors='coerce')

                medianas = X_train_num.median(numeric_only=True)
                X_train_num = X_train_num.fillna(medianas)
                X_val_num   = X_val_num.fillna(medianas)
                X_test_num  = X_test_num.fillna(medianas)
            else:
                X_train_num = pd.DataFrame(index=X_train.index)
                X_val_num   = pd.DataFrame(index=X_val.index)
                X_test_num  = pd.DataFrame(index=X_test.index)

            # 3) Reconstrói no MESMO ORDEM DE FEATURES
            X_train_proc = pd.concat([X_train_num, X_train_cat_enc], axis=1)[FEATURES]
            X_val_proc   = pd.concat([X_val_num,   X_val_cat_enc],   axis=1)[FEATURES]
            X_test_proc  = pd.concat([X_test_num,  X_test_cat_enc],  axis=1)[FEATURES]

            # --------- SELEÇÃO OPCIONAL DE FEATURES ---------
            USED_FEATURES = FEATURES
            keep_cols = select_top_features(X_train_proc, y_train, FEATURE_SUBSET_FRAC, random_state=RANDOM_STATE + fold)
            if keep_cols and len(keep_cols) < len(FEATURES):
                X_train_proc = X_train_proc[keep_cols]
                X_val_proc   = X_val_proc[keep_cols]
                X_test_proc  = X_test_proc[keep_cols]
                USED_FEATURES = keep_cols

            # --------- TREINO DO MODELO ---------
            model = RandomForestClassifier(**rf_params)
            model.fit(X_train_proc, y_train)

            # --------- PREVISÕES ---------
            # Probabilidades (binário assumido)
            y_proba_train_all = model.predict_proba(X_train_proc)
            y_proba_val_all   = model.predict_proba(X_val_proc)
            y_proba_test_all  = model.predict_proba(X_test_proc)

            y_proba_train_C1 = y_proba_train_all[:, 1]
            y_proba_val_C1   = y_proba_val_all[:, 1]
            y_proba_test_C1  = y_proba_test_all[:, 1]

            y_pred_train_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)
            y_pred_val_bin   = (y_proba_val_C1   >= THRESHOLD).astype(int)
            y_pred_test_bin  = (y_proba_test_C1  >= THRESHOLD).astype(int)

            # --------- GUARDAR PARTES COMPLETAS (com IDs) ---------
            # (mantém estrutura original; val não é salvo em CSV final por padrão)
            X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
            y_test_parts.append(pd.Series(y_test_, name='y_test').to_frame().assign(fold=fold).reset_index())
            y_test_pred_parts.append(
                pd.DataFrame(y_proba_test_all, columns=['prob_0', 'prob_1'], index=X_test_full.index)
                  .assign(fold=fold).reset_index()
            )

            X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
            y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
            y_train_pred_parts.append(
                pd.DataFrame(y_proba_train_all, columns=['prob_0', 'prob_1'], index=X_train_full.index)
                  .assign(fold=fold).reset_index()
            )

            # --------- MÉTRICAS (Treino) ---------
            cm_train = confusion_matrix(y_train, y_pred_train_bin)
            tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
            total_cm_treino = cm_train.sum()

            fpr_tr, tpr_tr, _ = roc_curve(y_train, y_proba_train_C1)
            fprs_train.append(fpr_tr); tprs_train.append(tpr_tr)
            auc_tr = roc_auc_score(y_train, y_proba_train_C1)
            aurocs_train.append(auc_tr)

            MASK_0_TRAIN = (y_train == 0)
            MASK_1_TRAIN = (y_train == 1)
            logloss_train_0 = log_loss(y_train[MASK_0_TRAIN], y_proba_train_C1[MASK_0_TRAIN],
                                       labels=[0,1]) if MASK_0_TRAIN.sum()>0 else np.nan
            logloss_train_1 = log_loss(y_train[MASK_1_TRAIN], y_proba_train_C1[MASK_1_TRAIN],
                                       labels=[0,1]) if MASK_1_TRAIN.sum()>0 else np.nan
            cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

            # --------- MÉTRICAS (Validação) ---------
            cm_val = confusion_matrix(y_val, y_pred_val_bin)
            tn_vl, fp_vl, fn_vl, tp_vl = cm_val.ravel()
            total_cm_val = cm_val.sum()

            auc_vl = roc_auc_score(y_val, y_proba_val_C1)
            MASK_0_VAL = (y_val == 0)
            MASK_1_VAL = (y_val == 1)
            logloss_val_0 = log_loss(y_val[MASK_0_VAL], y_proba_val_C1[MASK_0_VAL],
                                     labels=[0,1]) if MASK_0_VAL.sum()>0 else np.nan
            logloss_val_1 = log_loss(y_val[MASK_1_VAL], y_proba_val_C1[MASK_1_VAL],
                                     labels=[0,1]) if MASK_1_VAL.sum()>0 else np.nan
            cross_prop_val = pd.Series(y_val).value_counts(normalize=True) * 100

            # --------- MÉTRICAS (Teste) ---------
            cm_test = confusion_matrix(y_test_, y_pred_test_bin)
            tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
            total_cm_teste = cm_test.sum()

            fpr_ts, tpr_ts, _ = roc_curve(y_test_, y_proba_test_C1)
            fprs_test.append(fpr_ts); tprs_test.append(tpr_ts)
            auc_ts = roc_auc_score(y_test_, y_proba_test_C1)
            aurocs_test.append(auc_ts)

            MASK_0_TEST = (y_test_ == 0)
            MASK_1_TEST = (y_test_ == 1)
            logloss_test_0 = log_loss(y_test_[MASK_0_TEST], y_proba_test_C1[MASK_0_TEST],
                                      labels=[0,1]) if MASK_0_TEST.sum()>0 else np.nan
            logloss_test_1 = log_loss(y_test_[MASK_1_TEST], y_proba_test_C1[MASK_1_TEST],
                                      labels=[0,1]) if MASK_1_TEST.sum()>0 else np.nan
            cross_prop_test = pd.Series(y_test_).value_counts(normalize=True) * 100

            # --------- ARMAZENA RESULTADOS ---------
            resultados.append({
                'Conjunto': 'Treino',
                'Fold': fold,
                'Acurácia': accuracy_score(y_train, y_pred_train_bin),
                'Cross-Entropy C0': logloss_train_0,
                'Proporção C0': cross_prop_train.get(0, np.nan),
                'Cross-Entropy C1': logloss_train_1,
                'Proporção C1': cross_prop_train.get(1, np.nan),
                'F1 Score': f1_score(y_train, y_pred_train_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_train, y_pred_train_bin),
                'Precision': precision_score(y_train, y_pred_train_bin),
                'Recall': recall_score(y_train, y_pred_train_bin),
                'AUROC': auc_tr,
                'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
                'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
                'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
                'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
                'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            resultados.append({
                'Conjunto': 'Validação',
                'Fold': fold,
                'Acurácia': accuracy_score(y_val, y_pred_val_bin),
                'Cross-Entropy C0': logloss_val_0,
                'Proporção C0': cross_prop_val.get(0, np.nan),
                'Cross-Entropy C1': logloss_val_1,
                'Proporção C1': cross_prop_val.get(1, np.nan),
                'F1 Score': f1_score(y_val, y_pred_val_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_val, y_pred_val_bin),
                'Precision': precision_score(y_val, y_pred_val_bin),
                'Recall': recall_score(y_val, y_pred_val_bin),
                'AUROC': auc_vl,
                'TP': tp_vl, 'FP': fp_vl, 'TN': tn_vl, 'FN': fn_vl,
                'TP(%)': round(tp_vl / total_cm_val * 100, 2),
                'FP(%)': round(fp_vl / total_cm_val * 100, 2),
                'TN(%)': round(tn_vl / total_cm_val * 100, 2),
                'FN(%)': round(fn_vl / total_cm_val * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            resultados.append({
                'Conjunto': 'Teste',
                'Fold': fold,
                'Acurácia': accuracy_score(y_test_, y_pred_test_bin),
                'Cross-Entropy C0': logloss_test_0,
                'Proporção C0': cross_prop_test.get(0, np.nan),
                'Cross-Entropy C1': logloss_test_1,
                'Proporção C1': cross_prop_test.get(1, np.nan),
                'F1 Score': f1_score(y_test_, y_pred_test_bin),
                'Balanced Accuracy': balanced_accuracy_score(y_test_, y_pred_test_bin),
                'Precision': precision_score(y_test_, y_pred_test_bin),
                'Recall': recall_score(y_test_, y_pred_test_bin),
                'AUROC': auc_ts,
                'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
                'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
                'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
                'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
                'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
                'Tempo (s)': round(time.time() - t0, 2)
            })

            # --------- IMPORTÂNCIA DAS FEATURES (alinha com USED_FEATURES) ---------
            importancias = model.feature_importances_
            linha_importancia = {'Conjunto': 'Teste', 'Fold': fold}
            for nome, valor in zip(USED_FEATURES, importancias):
                linha_importancia[nome] = f"{valor * 100:.2f}%"
            importancias_lista.append(linha_importancia)

            # Salva modelo por "fold" (aqui fold=1)
            model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}_fold{fold}.pkl'
            joblib.dump(model, model_path)

            pbar.update(1)

# ================== PÓS-LOOP: SALVAMENTOS ==================
df_resultados = pd.DataFrame(resultados)

# adiciona linha de média das métricas numéricas
mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
mean_row['Conjunto'] = 'Média'
mean_row['Fold'] = 'Média'
df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# salva csvs principais (mantidos)
df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}.csv', index=False)
pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}.csv', index=False)

# Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
X_train_global = pd.concat(X_train_parts, ignore_index=True) if X_train_parts else pd.DataFrame()
y_train_global = pd.concat(y_train_parts, ignore_index=True) if y_train_parts else pd.DataFrame()
y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True) if y_train_pred_parts else pd.DataFrame()

X_test_global = pd.concat(X_test_parts, ignore_index=True) if X_test_parts else pd.DataFrame()
y_test_global = pd.concat(y_test_parts, ignore_index=True) if y_test_parts else pd.DataFrame()
y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True) if y_test_pred_parts else pd.DataFrame()

# Renomeia 'index' -> 'orig_index' (índice original)
for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
    if not df_block.empty and 'index' in df_block.columns:
        df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# Mesclas finais
if not X_train_global.empty and not y_train_global.empty:
    train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
    if not y_train_pred_global.empty:
        train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
else:
    train_df = pd.DataFrame()

if not X_test_global.empty and not y_test_global.empty:
    test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
    if not y_test_pred_global.empty:
        test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
else:
    test_df = pd.DataFrame()  # <- corrigido (antes: pdDataFrame())

# Adiciona y_proba / y_pred e salva (binário)
if not test_df.empty:
    test_df['y_proba'] = test_df['prob_1']
    test_df['y_pred']  = (test_df['y_proba'] >= THRESHOLD).astype(int)
    test_df.to_csv(CSV_OUT / f'rf_test_with_proba_{N_SPLITS}.csv', index=False)

if not train_df.empty:
    train_df['y_proba'] = train_df['prob_1']
    train_df['y_pred']  = (train_df['y_proba'] >= THRESHOLD).astype(int)
    train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}.csv', index=False)

# Dataset unificado (opcional)
if not test_df.empty:
    Path('../data/processed').mkdir(parents=True, exist_ok=True)
    test_df.to_csv(Path('../data/processed') / f'wdbc_rf_test_with_proba_{N_SPLITS}.csv', index=False)

# Augmenta o df original com previsões do TESTE (por orig_index)
try:
    if not test_df.empty:
        df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
        preds = test_df[['orig_index', 'y_pred', 'y_proba']]
        df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
        df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_{N_SPLITS}.csv', index=False)
    else:
        print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
except Exception as e_aug:
    print('Erro ao gerar dataset augmentado com previsões:', e_aug)

print('Resultados salvos em:', CSV_OUT)


In [ ]:
# # ================== IMPORTS ==================
# import os, time, joblib
# import numpy as np
# import pandas as pd
# from pathlib import Path

# from sklearn.model_selection import StratifiedKFold
# from sklearn.preprocessing import OrdinalEncoder
# from sklearn.metrics import (
#     accuracy_score, f1_score, balanced_accuracy_score, precision_score, recall_score,
#     roc_auc_score, roc_curve, confusion_matrix, log_loss
# )
# from sklearn.ensemble import RandomForestClassifier
# from matplotlib.backends.backend_pdf import PdfPages
# from tqdm import tqdm

# # ================== PREPARAÇÃO X, y ==================
# y = df['y']

# # Detecta colunas identificadoras a partir de EXCLUDE_COLUMNS (case-insensitive)
# exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
# ID_COLS = [c for c in df.columns if c.lower() in exclude_lower]
# ID_COLS = [c for c in ID_COLS if c != 'y']  # garante que não inclua a target

# # FEATURES são todas as colunas exceto ids e y
# FEATURES = [c for c in df.columns if c not in ID_COLS + ['y']]
# X_full = df[ID_COLS + FEATURES] if ID_COLS else df[FEATURES]

# # Detecta tipos (numéricas x categóricas) a partir de FEATURES
# num_cols = [c for c in FEATURES if pd.api.types.is_numeric_dtype(df[c])]
# cat_cols = [c for c in FEATURES if c not in num_cols]

# # K-Fold estratificado
# kf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)

# # ================== ESTRUTURAS DE RESULTADOS ==================
# resultados = []
# aurocs_train, aurocs_test = [], []
# fprs_train, tprs_train = [], []
# fprs_test,  tprs_test  = [], []
# importancias_lista = []

# X_train_parts, X_test_parts = [], []
# y_train_parts, y_test_parts = [], []
# y_train_pred_parts, y_test_pred_parts = [], []

# start_total = time.time()

# # ================== LOOP K-FOLD ==================
# with PdfPages(PDF_OUT / f'RF_CV_{N_SPLITS}.pdf') as pdf:
#     with tqdm(total=N_SPLITS, desc="🔄 Folds K-Fold") as pbar:
#         for fold, (train_idx, test_idx) in enumerate(kf.split(df[FEATURES], y), 1):
#             t0 = time.time()

#             # Partições completas (mantendo IDs p/ rastreabilidade)
#             X_train_full = X_full.iloc[train_idx].copy()
#             X_test_full  = X_full.iloc[test_idx].copy()

#             # Apenas as FEATURES para modelagem
#             X_train = X_train_full[FEATURES].copy()
#             X_test  = X_test_full[FEATURES].copy()
#             y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

#             # --------- PRÉ-PROCESSAMENTO POR FOLD (SEM VAZAMENTO) ---------
#             # 1) Categóricas → OrdinalEncoder (ajustado no treino)
#             if cat_cols:
#                 # Preenche NaN das categóricas com rótulo "Unknown" antes de encodar
#                 X_train_cat = X_train[cat_cols].astype('object').fillna('Unknown')
#                 X_test_cat  = X_test[cat_cols].astype('object').fillna('Unknown')

#                 oenc = OrdinalEncoder(
#                     handle_unknown='use_encoded_value',
#                     unknown_value=-1,
#                     encoded_missing_value=-1
#                 )
#                 X_train_cat_enc = pd.DataFrame(
#                     oenc.fit_transform(X_train_cat),
#                     columns=cat_cols, index=X_train.index
#                 )
#                 X_test_cat_enc = pd.DataFrame(
#                     oenc.transform(X_test_cat),
#                     columns=cat_cols, index=X_test.index
#                 )
#             else:
#                 X_train_cat_enc = pd.DataFrame(index=X_train.index)
#                 X_test_cat_enc  = pd.DataFrame(index=X_test.index)

#             # 2) Numéricas → coerção + imputação por mediana do TREINO
#             if num_cols:
#                 X_train_num = X_train[num_cols].apply(pd.to_numeric, errors='coerce')
#                 X_test_num  = X_test[num_cols].apply(pd.to_numeric, errors='coerce')

#                 medianas = X_train_num.median(numeric_only=True)
#                 X_train_num = X_train_num.fillna(medianas)
#                 X_test_num  = X_test_num.fillna(medianas)
#             else:
#                 X_train_num = pd.DataFrame(index=X_train.index)
#                 X_test_num  = pd.DataFrame(index=X_test.index)

#             # 3) Reconstrói X_train / X_test no MESMO ORDEM DE FEATURES
#             X_train_proc = pd.concat([X_train_num, X_train_cat_enc], axis=1)[FEATURES]
#             X_test_proc  = pd.concat([X_test_num,  X_test_cat_enc],  axis=1)[FEATURES]

#             # --------- TREINO DO MODELO ---------
#             model = RandomForestClassifier(**rf_params)
#             model.fit(X_train_proc, y_train)

#             # --------- PREVISÕES ---------
#             y_proba_test_all = model.predict_proba(X_test_proc)
#             y_proba_test_C1 = y_proba_test_all[:, 1]
#             y_pred_test_bin = (y_proba_test_C1 >= THRESHOLD).astype(int)

#             y_proba_train_all = model.predict_proba(X_train_proc)
#             y_proba_train_C1 = y_proba_train_all[:, 1]
#             y_pred_train_bin = (y_proba_train_C1 >= THRESHOLD).astype(int)

#             # --------- GUARDAR PARTES COMPLETAS (com IDs) ---------
#             X_test_parts.append(X_test_full.assign(fold=fold).reset_index())
#             y_test_parts.append(pd.Series(y_test, name='y_test').to_frame().assign(fold=fold).reset_index())
#             y_test_pred_parts.append(
#                 pd.DataFrame(y_proba_test_all, columns=['prob_0', 'prob_1'], index=X_test_full.index)
#                   .assign(fold=fold).reset_index()
#             )

#             X_train_parts.append(X_train_full.assign(fold=fold).reset_index())
#             y_train_parts.append(pd.Series(y_train, name='y_train').to_frame().assign(fold=fold).reset_index())
#             y_train_pred_parts.append(
#                 pd.DataFrame(y_proba_train_all, columns=['prob_0', 'prob_1'], index=X_train_full.index)
#                   .assign(fold=fold).reset_index()
#             )

#             # --------- MÉTRICAS (Treino) ---------
#             cm_train = confusion_matrix(y_train, y_pred_train_bin)
#             tn_tr, fp_tr, fn_tr, tp_tr = cm_train.ravel()
#             total_cm_treino = cm_train.sum()

#             fpr_tr, tpr_tr, _ = roc_curve(y_train, y_proba_train_C1)
#             fprs_train.append(fpr_tr); tprs_train.append(tpr_tr)
#             auc_tr = roc_auc_score(y_train, y_proba_train_C1)
#             aurocs_train.append(auc_tr)

#             MASK_0_TRAIN = (y_train == 0)
#             MASK_1_TRAIN = (y_train == 1)
#             logloss_train_0 = log_loss(y_train[MASK_0_TRAIN], y_proba_train_C1[MASK_0_TRAIN],
#                                        labels=[0,1]) if MASK_0_TRAIN.sum()>0 else np.nan
#             logloss_train_1 = log_loss(y_train[MASK_1_TRAIN], y_proba_train_C1[MASK_1_TRAIN],
#                                        labels=[0,1]) if MASK_1_TRAIN.sum()>0 else np.nan
#             cross_prop_train = pd.Series(y_train).value_counts(normalize=True) * 100

#             # --------- MÉTRICAS (Teste) ---------
#             cm_test = confusion_matrix(y_test, y_pred_test_bin)
#             tn_ts, fp_ts, fn_ts, tp_ts = cm_test.ravel()
#             total_cm_teste = cm_test.sum()

#             fpr_ts, tpr_ts, _ = roc_curve(y_test, y_proba_test_C1)
#             fprs_test.append(fpr_ts); tprs_test.append(tpr_ts)
#             auc_ts = roc_auc_score(y_test, y_proba_test_C1)
#             aurocs_test.append(auc_ts)

#             MASK_0_TEST = (y_test == 0)
#             MASK_1_TEST = (y_test == 1)
#             logloss_test_0 = log_loss(y_test[MASK_0_TEST], y_proba_test_C1[MASK_0_TEST],
#                                       labels=[0,1]) if MASK_0_TEST.sum()>0 else np.nan
#             logloss_test_1 = log_loss(y_test[MASK_1_TEST], y_proba_test_C1[MASK_1_TEST],
#                                       labels=[0,1]) if MASK_1_TEST.sum()>0 else np.nan
#             cross_prop_test = pd.Series(y_test).value_counts(normalize=True) * 100

#             # --------- ARMAZENA RESULTADOS ---------
#             resultados.append({
#                 'Conjunto': 'Treino',
#                 'Fold': fold,
#                 'Acurácia': accuracy_score(y_train, y_pred_train_bin),
#                 'Cross-Entropy C0': logloss_train_0,
#                 'Proporção C0': cross_prop_train.get(0, np.nan),
#                 'Cross-Entropy C1': logloss_train_1,
#                 'Proporção C1': cross_prop_train.get(1, np.nan),
#                 'F1 Score': f1_score(y_train, y_pred_train_bin),
#                 'Balanced Accuracy': balanced_accuracy_score(y_train, y_pred_train_bin),
#                 'Precision': precision_score(y_train, y_pred_train_bin),
#                 'Recall': recall_score(y_train, y_pred_train_bin),
#                 'AUROC': auc_tr,
#                 'TP': tp_tr, 'FP': fp_tr, 'TN': tn_tr, 'FN': fn_tr,
#                 'TP(%)': round(tp_tr / total_cm_treino * 100, 2),
#                 'FP(%)': round(fp_tr / total_cm_treino * 100, 2),
#                 'TN(%)': round(tn_tr / total_cm_treino * 100, 2),
#                 'FN(%)': round(fn_tr / total_cm_treino * 100, 2),
#                 'Tempo (s)': round(time.time() - t0, 2)
#             })

#             resultados.append({
#                 'Conjunto': 'Teste',
#                 'Fold': fold,
#                 'Acurácia': accuracy_score(y_test, y_pred_test_bin),
#                 'Cross-Entropy C0': logloss_test_0,
#                 'Proporção C0': cross_prop_test.get(0, np.nan),
#                 'Cross-Entropy C1': logloss_test_1,
#                 'Proporção C1': cross_prop_test.get(1, np.nan),
#                 'F1 Score': f1_score(y_test, y_pred_test_bin),
#                 'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred_test_bin),
#                 'Precision': precision_score(y_test, y_pred_test_bin),
#                 'Recall': recall_score(y_test, y_pred_test_bin),
#                 'AUROC': auc_ts,
#                 'TP': tp_ts, 'FP': fp_ts, 'TN': tn_ts, 'FN': fn_ts,
#                 'TP(%)': round(tp_ts / total_cm_teste * 100, 2),
#                 'FP(%)': round(fp_ts / total_cm_teste * 100, 2),
#                 'TN(%)': round(tn_ts / total_cm_teste * 100, 2),
#                 'FN(%)': round(fn_ts / total_cm_teste * 100, 2),
#                 'Tempo (s)': round(time.time() - t0, 2)
#             })

#             # --------- IMPORTÂNCIA DAS FEATURES (alinha com FEATURES) ---------
#             importancias = model.feature_importances_
#             linha_importancia = {'Conjunto': 'Teste', 'Fold': fold}
#             for nome, valor in zip(FEATURES, importancias):
#                 linha_importancia[nome] = f"{valor * 100:.2f}%"
#             importancias_lista.append(linha_importancia)

#             # Salva modelo por fold (opcional)
#             model_path = MODEL_DIR / f'rf_model_cf{N_SPLITS}_fold{fold}.pkl'
#             joblib.dump(model, model_path)

#             pbar.update(1)

# # ================== PÓS-LOOP: SALVAMENTOS ==================
# df_resultados = pd.DataFrame(resultados)

# # adiciona linha de média das métricas numéricas
# mean_row = df_resultados.select_dtypes(include=[np.number]).mean()
# mean_row['Conjunto'] = 'Média'
# mean_row['Fold'] = 'Média'
# df_resultados = pd.concat([df_resultados, pd.DataFrame([mean_row])], ignore_index=True, sort=False)

# # salva csvs principais
# df_resultados.to_csv(CSV_OUT / f'rf_cv_results_{N_SPLITS}.csv', index=False)
# pd.DataFrame(importancias_lista).to_csv(CSV_OUT / f'rf_feature_importances_{N_SPLITS}.csv', index=False)

# # Concatena blocos para gerar dataframes completos de treino/teste com probabilidades
# X_train_global = pd.concat(X_train_parts, ignore_index=True) if X_train_parts else pd.DataFrame()
# y_train_global = pd.concat(y_train_parts, ignore_index=True) if y_train_parts else pd.DataFrame()
# y_train_pred_global = pd.concat(y_train_pred_parts, ignore_index=True) if y_train_pred_parts else pd.DataFrame()

# X_test_global = pd.concat(X_test_parts, ignore_index=True) if X_test_parts else pd.DataFrame()
# y_test_global = pd.concat(y_test_parts, ignore_index=True) if y_test_parts else pd.DataFrame()
# y_test_pred_global = pd.concat(y_test_pred_parts, ignore_index=True) if y_test_pred_parts else pd.DataFrame()

# # Renomeia 'index' -> 'orig_index' (índice original)
# for df_block in [X_train_global, X_test_global, y_train_global, y_test_global, y_train_pred_global, y_test_pred_global]:
#     if not df_block.empty and 'index' in df_block.columns:
#         df_block.rename(columns={'index': 'orig_index'}, inplace=True)

# # Mesclas finais
# if not X_train_global.empty and not y_train_global.empty:
#     train_df = X_train_global.merge(y_train_global, on=['orig_index', 'fold'], how='left')
#     if not y_train_pred_global.empty:
#         train_df = train_df.merge(y_train_pred_global, on=['orig_index', 'fold'], how='left')
# else:
#     train_df = pd.DataFrame()

# if not X_test_global.empty and not y_test_global.empty:
#     test_df = X_test_global.merge(y_test_global, on=['orig_index', 'fold'], how='left')
#     if not y_test_pred_global.empty:
#         test_df = test_df.merge(y_test_pred_global, on=['orig_index', 'fold'], how='left')
# else:
#     test_df = pd.DataFrame()

# # Adiciona y_proba / y_pred e salva
# if not test_df.empty:
#     test_df['y_proba'] = test_df['prob_1']
#     test_df['y_pred']  = (test_df['y_proba'] >= THRESHOLD).astype(int)
#     test_df.to_csv(CSV_OUT / f'rf_test_with_proba_{N_SPLITS}.csv', index=False)

# if not train_df.empty:
#     train_df['y_proba'] = train_df['prob_1']
#     train_df['y_pred']  = (train_df['y_proba'] >= THRESHOLD).astype(int)
#     train_df.to_csv(CSV_OUT / f'rf_train_with_proba_{N_SPLITS}.csv', index=False)

# # Dataset unificado (opcional)
# if not test_df.empty:
#     Path('../data/processed').mkdir(parents=True, exist_ok=True)
#     test_df.to_csv(Path('../data/processed') / f'wdbc_rf_test_with_proba_{N_SPLITS}.csv', index=False)

# # Augmenta o df original com previsões do TESTE (por orig_index)
# try:
#     if not test_df.empty:
#         df_aug = df.reset_index().rename(columns={'index': 'orig_index'})
#         preds = test_df[['orig_index', 'y_pred', 'y_proba']]
#         df_augmented = pd.merge(df_aug, preds, how='left', on='orig_index')
#         df_augmented.to_csv(CSV_OUT / f'database_used_{DATASET_NAME}_with_preds_{N_SPLITS}.csv', index=False)
#     else:
#         print('test_df vazio: não foi possível gerar dataset augmentado com previsões')
# except Exception as e_aug:
#     print('Erro ao gerar dataset augmentado com previsões:', e_aug)

# print('Resultados salvos em:', CSV_OUT)


In [ ]:
# # Diagnóstico rápido para NaN / tipos em train_df e test_df
# for name, ycol in [('train_df', 'y_train'), ('test_df', 'y_test')]:
#     df_block = globals().get(name)
#     if df_block is None:
#         print(f"{name} não existe")
#         continue
#     if df_block.empty:
#         print(f"{name} existe mas está vazio")
#         continue
#     print(f"\n{name}: shape={df_block.shape}")
#     for col in [ycol, 'prob_1', 'y_proba']:
#         if col in df_block.columns:
#             print(f"  {col}: nulls={df_block[col].isnull().sum()}, dtype={df_block[col].dtype}, unique_count={df_block[col].nunique(dropna=True)}")
#     cols_to_show = [c for c in [ycol, 'prob_1', 'y_proba'] if c in df_block.columns]
#     display(df_block[cols_to_show].head(10))

In [ ]:
from sklearn.metrics import auc  # necessário para usar auc(fpr, tpr)

# Geração de PDF resumo (com algumas páginas importantes)
with PdfPages(PDF_OUT / f'RF_CV_SUMMARY_{N_SPLITS}.pdf') as pdf:
    # ---------------- Página de parâmetros ----------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.axis('off')
    parametros = rf_params.copy()
    parametros['threshold'] = THRESHOLD
    parametros['n_splits'] = N_SPLITS
    parametros['split_strategy'] = '8/1/1'  # <<< ajuste mínimo p/ refletir o esquema
    texto = 'Algoritmo: Random Forest\n' + '\n'.join([f'{k}: {v}' for k, v in parametros.items()])
    ax.text(0, 1, texto, fontsize=11, family='monospace', verticalalignment='top')
    ax.set_title('📌 Parâmetros do Modelo - Random Forest')
    try:
        png_path = IMAGES_OUT / f'params_{N_SPLITS}.png'
        fig.savefig(png_path, dpi=300, bbox_inches='tight')
    except Exception:
        pass
    pdf.savefig(fig)
    plt.close()

    # ---------------- Página de métricas resumidas ----------------
    if not df_resultados.empty:
        fig, ax = plt.subplots(figsize=(12, 6))
        ax.axis('off')
        # agrupa por Conjunto (Treino / Validação / Teste) – já compatível com 8/1/1
        display_df = df_resultados.groupby(['Conjunto']).mean(numeric_only=True).T.round(4)
        table = ax.table(cellText=display_df.values,
                         colLabels=display_df.columns,
                         rowLabels=display_df.index,
                         loc='center')
        table.auto_set_font_size(False)
        table.set_fontsize(9)
        try:
            png_path = IMAGES_OUT / f'metrics_summary_{N_SPLITS}.png'
            fig.savefig(png_path, dpi=300, bbox_inches='tight')
        except Exception:
            pass
        pdf.savefig(fig)
        plt.close()

    # ---------------- Página ROC - Treino vs Teste (lado a lado) ----------------
    try:
        plotted = False

        # ROC Treino
        if 'y_train' in train_df.columns and 'prob_1' in train_df.columns:
            y_train_all = train_df['y_train']
            y_score_train_all = train_df['prob_1']
            fpr_train, tpr_train, _ = roc_curve(y_train_all, y_score_train_all)
            roc_auc_train = auc(fpr_train, tpr_train)
            plotted = True
        elif len(fprs_train) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_train, tprs_train):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_train, tpr_train = mean_fpr, mean_tpr
            roc_auc_train = np.mean(aurocs_train) if len(aurocs_train) > 0 else auc(fpr_train, tpr_train)
            plotted = True

        # ROC Teste
        if 'y_test' in test_df.columns and 'prob_1' in test_df.columns:
            y_test_all = test_df['y_test']
            y_score_test_all = test_df['prob_1']
            fpr_test, tpr_test, _ = roc_curve(y_test_all, y_score_test_all)
            roc_auc_test = auc(fpr_test, tpr_test)
            plotted = True
        elif len(fprs_test) > 0:
            mean_fpr = np.linspace(0, 1, 200)
            tprs = []
            for fpr, tpr in zip(fprs_test, tprs_test):
                tpr_i = np.interp(mean_fpr, fpr, tpr)
                tpr_i[0] = 0.0
                tprs.append(tpr_i)
            mean_tpr = np.mean(tprs, axis=0)
            mean_tpr[-1] = 1.0
            fpr_test, tpr_test = mean_fpr, mean_tpr
            roc_auc_test = np.mean(aurocs_test) if len(aurocs_test) > 0 else auc(fpr_test, tpr_test)
            plotted = True

        if plotted:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))
            # ROC Treino
            try:
                axes[0].plot(fpr_train, tpr_train, color='blue', label=f'AUC = {roc_auc_train:.4f}')
                axes[0].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[0].set_title('ROC - Treino (80%)')
                axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
                axes[0].legend(loc='lower right')
            except Exception:
                axes[0].text(0.5, 0.5, 'Não há dados de treino para ROC', ha='center')
            # ROC Teste
            try:
                axes[1].plot(fpr_test, tpr_test, color='green', label=f'AUC = {roc_auc_test:.4f}')
                axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
                axes[1].set_title('ROC - Teste (10%)')
                axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')
                axes[1].legend(loc='lower right')
            except Exception:
                axes[1].text(0.5, 0.5, 'Não há dados de teste para ROC', ha='center')
            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'roc_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
        else:
            print('Não foi possível gerar curvas ROC (dados ausentes)')
    except Exception as e:
        print('Não foi possível gerar ROC agregado:', e)

    # ---------------- Página Confusion Matrix - Treino vs Teste ----------------
    try:
        def build_cm_from_df(df_block, true_col, pred_col):
            y_true = df_block[true_col]
            y_pred = df_block[pred_col]
            cm = confusion_matrix(y_true, y_pred)
            total = cm.sum()
            cm_percent = cm / total * 100 if total > 0 else np.zeros_like(cm, dtype=float)
            return cm, cm_percent

        cm_train = cm_train_pct = cm_test = cm_test_pct = None

        if 'y_train' in train_df.columns and 'y_pred' in train_df.columns:
            cm_train, cm_train_pct = build_cm_from_df(train_df, 'y_train', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                train_rows = df_resultados[df_resultados['Conjunto'] == 'Treino']
                TP = int(train_rows['TP'].sum()) if not train_rows.empty else 0
                FP = int(train_rows['FP'].sum()) if not train_rows.empty else 0
                TN = int(train_rows['TN'].sum()) if not train_rows.empty else 0
                FN = int(train_rows['FN'].sum()) if not train_rows.empty else 0
                cm_train = np.array([[TN, FP], [FN, TP]])
                total = cm_train.sum()
                cm_train_pct = cm_train / total * 100 if total > 0 else np.zeros_like(cm_train, dtype=float)

        if 'y_test' in test_df.columns and 'y_pred' in test_df.columns:
            cm_test, cm_test_pct = build_cm_from_df(test_df, 'y_test', 'y_pred')
        else:
            if 'TP' in df_resultados.columns:
                test_rows = df_resultados[df_resultados['Conjunto'] == 'Teste']
                TP = int(test_rows['TP'].sum()) if not test_rows.empty else 0
                FP = int(test_rows['FP'].sum()) if not test_rows.empty else 0
                TN = int(test_rows['TN'].sum()) if not test_rows.empty else 0
                FN = int(test_rows['FN'].sum()) if not test_rows.empty else 0
                cm_test = np.array([[TN, FP], [FN, TP]])
                total = cm_test.sum()
                cm_test_pct = cm_test / total * 100 if total > 0 else np.zeros_like(cm_test, dtype=float)

        if (cm_train is None and cm_test is None):
            print('Não há dados suficientes para gerar matrizes de confusão.')
        else:
            fig, axes = plt.subplots(1, 2, figsize=(12, 5))

            def plot_cm(ax, cm, cm_pct, title):
                if cm is None:
                    ax.text(0.5, 0.5, 'Sem dados', ha='center', va='center'); ax.axis('off'); return
                annot = np.empty(cm.shape, dtype=object)
                for i in range(cm.shape[0]):
                    for j in range(cm.shape[1]):
                        annot[i, j] = f"{int(cm[i,j])}\n({cm_pct[i,j]:.2f}%)"
                sns.heatmap(cm, annot=annot, fmt='', cmap='Blues', ax=ax, cbar=False, annot_kws={'size':12})
                ax.set_xlabel('Predito'); ax.set_ylabel('Verdadeiro')
                ax.set_xticklabels(['0','1']); ax.set_yticklabels(['0','1'])
                ax.set_title(title)

            plot_cm(axes[0], cm_train, cm_train_pct, 'Matriz de Confusão - Treino (80%)')
            plot_cm(axes[1], cm_test, cm_test_pct, 'Matriz de Confusão - Teste (10%)')

            plt.tight_layout()
            try:
                png_path = IMAGES_OUT / f'confusion_train_test_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight')
            plt.close()
    except Exception as e:
        print('Não foi possível gerar matrizes de confusão:', e)

    # ---------------- Importâncias médias + Pareto ----------------
    try:
        df_imp = pd.DataFrame(importancias_lista).replace('%', '', regex=True)
        cols = [c for c in df_imp.columns if c not in ['Conjunto', 'Fold']]
        try:
            exclude_lower = {c.lower() for c in EXCLUDE_COLUMNS}
        except Exception:
            exclude_lower = set()
        cols = [c for c in cols if c.lower() not in exclude_lower]

        if len(cols) == 0:
            print('Nenhuma feature disponível para plotar importâncias (após excluir ids).')
        else:
            df_imp_mean = df_imp[cols].astype(float).mean().sort_values(ascending=True)

            height = max(6, 0.25 * len(df_imp_mean))
            fig, ax = plt.subplots(figsize=(10, height))
            df_imp_mean.plot(kind='barh', ax=ax, color='tab:blue')
            ax.set_xlabel('Importância média (%)')
            ax.set_title('Importância média das features (porcentagem)')
            plt.tight_layout(); plt.subplots_adjust(left=0.30)
            try:
                png_path = IMAGES_OUT / f'feature_importances_{N_SPLITS}.png'
                fig.savefig(png_path, dpi=300, bbox_inches='tight')
            except Exception:
                pass
            pdf.savefig(fig, bbox_inches='tight'); plt.close()

            # Tabela de importâncias
            try:
                table_df = df_imp_mean.sort_values(ascending=False).to_frame(name='Importance (%)').round(2)
                table_height = max(4, 0.3 * len(table_df))
                fig, ax = plt.subplots(figsize=(8, table_height))
                ax.axis('off')
                table = ax.table(cellText=table_df.values,
                                 colLabels=table_df.columns,
                                 rowLabels=table_df.index,
                                 loc='center')
                table.auto_set_font_size(False); table.set_fontsize(9); table.scale(1, 1.2)
                ax.set_title('Tabela: Importância média das features (%)', pad=20)
                plt.tight_layout()
                pdf.savefig(fig, bbox_inches='tight'); plt.close()
            except Exception as e_table:
                print('Não foi possível gerar tabela de importâncias:', e_table)

            # --- Pareto 80/20 ---
            try:
                imp_desc = df_imp_mean.sort_values(ascending=False)
                pareto_df = imp_desc.to_frame(name='importance_pct')
                pareto_df['cumulative_pct'] = pareto_df['importance_pct'].cumsum()
                threshold_pct = (PARETO_THRESHOLD if 'PARETO_THRESHOLD' in globals() else 0.8) * 100

                import numpy as _np
                pos = _np.searchsorted(pareto_df['cumulative_pct'].values, threshold_pct)
                if pos < len(pareto_df):
                    selected_features = pareto_df.iloc[:pos+1].index.tolist()
                else:
                    selected_features = pareto_df.index.tolist()

                PARETO_SELECTED_FEATURES = selected_features
                PARETO_SELECTED_IMPORTANCES = imp_desc.loc[selected_features].round(2).to_dict()

                pareto_df.reset_index().rename(columns={'index': 'feature'}) \
                         .to_csv(CSV_OUT / f'pareto_importances_{int((PARETO_THRESHOLD*100))}.csv', index=False)
                pd.DataFrame({'feature': PARETO_SELECTED_FEATURES,
                              'importance_pct': [PARETO_SELECTED_IMPORTANCES[f] for f in PARETO_SELECTED_FEATURES]}) \
                  .to_csv(CSV_OUT / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.csv', index=False)

                try:
                    import json
                    with open(MODEL_DIR / f'pareto_selected_features_{int((PARETO_THRESHOLD*100))}.json', 'w') as fh:
                        json.dump({'threshold': PARETO_THRESHOLD,
                                   'selected_features': PARETO_SELECTED_FEATURES,
                                   'importances': PARETO_SELECTED_IMPORTANCES}, fh, indent=2)
                except Exception:
                    pass

                # Gráfico Pareto
                try:
                    fig, ax1 = plt.subplots(figsize=(10, 6))
                    x = pareto_df.index.astype(str)
                    ax1.bar(x, pareto_df['importance_pct'], color='tab:blue')
                    ax1.set_ylabel('Importância (%)')
                    ax1.set_xticklabels(x, rotation=90)
                    ax2 = ax1.twinx()
                    ax2.plot(x, pareto_df['cumulative_pct'], color='red', marker='o')
                    ax2.axhline(threshold_pct, color='gray', linestyle='--')
                    ax2.set_ylabel('Cumulative (%)')
                    ax1.set_title(f'Pareto (threshold={int(threshold_pct)}%) - selecionadas: {len(PARETO_SELECTED_FEATURES)}')
                    plt.tight_layout()
                    try:
                        png_path = IMAGES_OUT / f'pareto_{int((PARETO_THRESHOLD*100))}.png'
                        fig.savefig(png_path, dpi=300, bbox_inches='tight')
                    except Exception:
                        pass
                    pdf.savefig(fig, bbox_inches='tight'); plt.close()
                except Exception as e_plot:
                    print('Não foi possível gerar gráfico Pareto:', e_plot)

            except Exception as e_pareto:
                print('Não foi possível gerar análise Pareto:', e_pareto)

            # --- Tabela de variâncias: Treino vs Teste (APENAS NUMÉRICAS) ---
            try:
                if 'FEATURES' in globals() and not (train_df.empty or test_df.empty):
                    feats_common = [f for f in FEATURES if f in train_df.columns and f in test_df.columns]
                    if len(feats_common) == 0:
                        print('Nenhuma feature comum entre train/test para calcular variância.')
                    else:
                        train_num = train_df[feats_common].apply(pd.to_numeric, errors='coerce')
                        test_num  = test_df[feats_common].apply(pd.to_numeric, errors='coerce')

                        valid_cols = [c for c in feats_common if train_num[c].notna().any() and test_num[c].notna().any()]
                        if len(valid_cols) == 0:
                            print('Nenhuma feature numérica válida em comum para variância.')
                        else:
                            var_train = train_num[valid_cols].var(ddof=1, numeric_only=True)
                            var_test  = test_num[valid_cols].var(ddof=1, numeric_only=True)
                            var_df = pd.DataFrame({'var_train': var_train, 'var_test': var_test})
                            var_df['diff'] = var_df['var_train'] - var_df['var_test']
                            var_df['pct_diff'] = (var_df['diff'].abs() / var_df['var_train'].replace({0: np.nan})) * 100

                            try:
                                thresh_pct = VAR_DIFF_PCT_THRESHOLD * 100
                            except Exception:
                                thresh_pct = 10.0
                            var_df['high_low'] = var_df['pct_diff'].apply(
                                lambda x: 'High' if (pd.notna(x) and x >= thresh_pct) else 'Low'
                            )
                            var_df = var_df.sort_values('var_train', ascending=False)

                            out_var_csv = CSV_OUT / 'feature_variances_train_test.csv'
                            var_df.reset_index().rename(columns={'index': 'feature'}).to_csv(out_var_csv, index=False)

                            # Página com a tabela
                            try:
                                max_rows = len(var_df)
                                fig_h = min(12, 0.25 * max_rows + 2)
                                fig, ax = plt.subplots(figsize=(11.69, fig_h))
                                ax.axis('off')
                                tbl = ax.table(
                                    cellText=var_df.round(4).reset_index().values,
                                    colLabels=['feature', 'var_train', 'var_test', 'diff', 'pct_diff', 'high_low'],
                                    loc='center'
                                )
                                tbl.auto_set_font_size(False)
                                tbl.set_fontsize(8)
                                tbl.scale(1, 1.2)
                                ax.set_title('Tabela: Variância das Features (Treino vs Teste)')
                                plt.tight_layout()
                                pdf.savefig(fig, bbox_inches='tight'); plt.close()
                            except Exception as e_tbl:
                                print('Não foi possível gerar página PDF da tabela de variâncias:', e_tbl)
                else:
                    print('Dados de treino/teste ou FEATURES ausentes; não foi possível calcular variâncias.')
            except Exception as e_var:
                print('Erro ao calcular tabela de variâncias:', e_var)

            # --- Gráficos de violino para as features selecionadas pelo Pareto ---
            try:
                if (PARETO_SELECTED_FEATURES is None) or (len(PARETO_SELECTED_FEATURES) == 0):
                    print('Nenhuma feature selecionada pelo Pareto para gerar violinos.')
                elif train_df.empty or test_df.empty:
                    print('Dados de treino ou teste ausentes para gerar violinos.')
                else:
                    violin_features = [f for f in PARETO_SELECTED_FEATURES if f in train_df.columns and f in test_df.columns]
                    if len(violin_features) == 0:
                        print('Nenhuma das features selecionadas pelo Pareto está presente em train/test para plotagem.')
                    else:
                        combined_list = []
                        train_sub = train_df[violin_features].copy(); train_sub['__set__'] = 'train'
                        test_sub  = test_df[violin_features].copy();  test_sub['__set__'] = 'test'
                        combined = pd.concat([train_sub, test_sub], ignore_index=True)

                        per_page = 6
                        import math
                        n = len(violin_features)
                        pages = math.ceil(n / per_page)
                        for p in range(pages):
                            start = p * per_page
                            end = min(start + per_page, n)
                            page_feats = violin_features[start:end]
                            rows = 3
                            fig, axes = plt.subplots(rows, 2, figsize=(8.27, 11.69))  # A4 portrait
                            axes = axes.flatten()
                            for i, feat in enumerate(page_feats):
                                ax = axes[i]
                                sns.violinplot(x='__set__', y=feat, data=combined,
                                               order=['train', 'test'], ax=ax,
                                               inner='quartile', palette=['#4c72b0', '#55a868'])
                                ax.set_title(f'Violin: {feat} (train vs test)')
                                ax.set_xlabel('')
                            for j in range(len(page_feats), len(axes)):
                                axes[j].axis('off')
                            plt.tight_layout()
                            pdf.savefig(fig, bbox_inches='tight')

                            # imagens individuais
                            try:
                                for feat in page_feats:
                                    fig_f, ax_f = plt.subplots(figsize=(6, 4))
                                    sns.violinplot(x='__set__', y=feat, data=combined,
                                                   order=['train', 'test'], ax=ax_f,
                                                   inner='quartile', palette=['#4c72b0', '#55a868'])
                                    ax_f.set_title(f'Violin: {feat} (train vs test)')
                                    ax_f.set_xlabel('')
                                    plt.tight_layout()
                                    png_path = IMAGES_OUT / f'violin_{feat}.png'
                                    fig_f.savefig(png_path, dpi=300, bbox_inches='tight')
                                    plt.close(fig_f)
                            except Exception:
                                pass
                            plt.close(fig)
            except Exception as e_violin:
                print('Não foi possível gerar gráficos de violino:', e_violin)
    except Exception as e:
        print('Não foi possível gerar gráfico de importâncias:', e)

print('PDF resumo gerado em:', PDF_OUT)


In [ ]:
png_path

In [ ]:
# Salvar X_test_basic_full.csv, X_train_basic_full.csv e X_validation_basic_full.csv
# com todas as colunas do arquivo cru + y, y_pred, y_proba
import pandas as pd
from pathlib import Path

# --- caminhos/pastas ---
# DATASET_PATH deve apontar para o CSV "cru" original
assert 'DATASET_PATH' in globals(), "Defina DATASET_PATH apontando para o CSV original."
BASICO_CSV = BASICO_CSV if 'BASICO_CSV' in globals() else Path('basics_csv')
BASICO_CSV.mkdir(parents=True, exist_ok=True)

# --- carregar arquivo cru ---
raw_df = pd.read_csv(DATASET_PATH)

# Para FHS (e similares), se existir a coluna 'stroke', usar como y (sem sobrescrever se já existir)
if 'stroke' in raw_df.columns and 'y' not in raw_df.columns:
    raw_df['y'] = raw_df['stroke']

# --- função utilitária para salvar splits alinhando por índice ---
def save_full_split(indices, y_true, y_pred, y_proba, split_name: str):
    """
    indices: índice original das linhas (use X_*_full.index para manter alinhamento com raw_df)
    y_true / y_pred / y_proba: arrays ou Series com MESMO comprimento de indices
    split_name: 'train', 'validation' ou 'test'
    """
    df_full = raw_df.loc[indices].copy()

    y_true_s  = pd.Series(y_true,  index=indices, name='y')
    y_pred_s  = pd.Series(y_pred,  index=indices, name='y_pred')
    y_proba_s = pd.Series(y_proba, index=indices, name='y_proba')

    df_full = df_full.assign(y=y_true_s, y_pred=y_pred_s, y_proba=y_proba_s)

    out_path = BASICO_CSV / f'X_{split_name}_basic_full.csv'
    df_full.to_csv(out_path, index=True)
    print(f'✅ Arquivo salvo: {out_path} (linhas: {len(df_full)})')

# =========================
# CHAMADAS (no ponto em que as variáveis existem no fluxo 8/1/1)
# Use *_full.index para preservar o índice original do raw_df
# =========================
# y_proba_*_C1: probabilidades da classe positiva
# y_pred_*_bin: predições binárias

# --- Teste (10%) ---
save_full_split(
    indices=X_test_full.index,
    y_true=y_test,                 # ou y_test_ conforme seu script
    y_pred=y_pred_test_bin,
    y_proba=y_proba_test_C1,
    split_name='test'
)

# --- Treino (80%) ---
save_full_split(
    indices=X_train_full.index,
    y_true=y_train,
    y_pred=y_pred_train_bin,
    y_proba=y_proba_train_C1,
    split_name='train'
)

# --- Validação (10%) — novo no 8/1/1 ---
# Chame esta parte somente se você tiver X_val_full / y_val / y_pred_val_bin / y_proba_val_C1 definidos
if 'X_val_full' in globals() and 'y_val' in globals() and \
   'y_pred_val_bin' in globals() and 'y_proba_val_C1' in globals():
    save_full_split(
        indices=X_val_full.index,
        y_true=y_val,
        y_pred=y_pred_val_bin,
        y_proba=y_proba_val_C1,
        split_name='validation'
    )


In [ ]:
# ==========================================================
# Salvar X_test_basic_full.csv, X_train_basic_full.csv
# (e, se disponível, X_validation_basic_full.csv)
# com todas as colunas do arquivo cru + y, y_pred, y_proba
# ==========================================================
import pandas as pd
from pathlib import Path

# Caminho e diretórios
BASICO_CSV = Path("../model_reports/random_forest/basico/csv")
BASICO_CSV.mkdir(parents=True, exist_ok=True)

# Carregar o arquivo cru original
raw_df = pd.read_csv(DATASET_PATH)

# Se existir a coluna 'stroke', criar 'y'
if 'stroke' in raw_df.columns and 'y' not in raw_df.columns:
    raw_df['y'] = raw_df['stroke']

# Função para salvar os splits completos
def save_full_split(indices, y_true, y_pred, y_proba, split_name):
    df_full = raw_df.loc[indices].copy()

    # Garante alinhamento correto
    y_true_s  = pd.Series(y_true,  index=indices, name='y')
    y_pred_s  = pd.Series(y_pred,  index=indices, name='y_pred')
    y_proba_s = pd.Series(y_proba, index=indices, name='y_proba')

    df_full = df_full.assign(y=y_true_s, y_pred=y_pred_s, y_proba=y_proba_s)

    out_path = BASICO_CSV / f"X_{split_name}_basic_full.csv"
    df_full.to_csv(out_path, index=True)
    print(f"✅ Arquivo salvo: {out_path} ({len(df_full)} linhas)")

# --- Teste (10%) ---
save_full_split(
    indices=X_test_full.index,
    y_true=y_test,
    y_pred=y_pred_test_bin,
    y_proba=y_proba_test_C1,   # variável correta!
    split_name="test"
)

# --- Treino (80%) ---
save_full_split(
    indices=X_train_full.index,
    y_true=y_train,
    y_pred=y_pred_train_bin,
    y_proba=y_proba_train_C1,  # variável correta!
    split_name="train"
)

# --- Validação (10%) — opcional no 8/1/1 (somente se existir no fluxo) ---
if all(v in globals() for v in ["X_val_full", "y_val", "y_pred_val_bin", "y_proba_val_C1"]):
    save_full_split(
        indices=X_val_full.index,
        y_true=y_val,
        y_pred=y_pred_val_bin,
        y_proba=y_proba_val_C1,
        split_name="validation"
    )
